# `download_com_antitrust_v14` — European Commission AT case manifest builder

This notebook converts the European Commission antitrust case JSON file into Excel-friendly manifest tables.

## Data sources

There are two practical ways to obtain the underlying Commission case data:

1. **European Commission Competition Case Search**  
   Cases can be searched here: https://competition-cases.ec.europa.eu/search?caseInstrument=AT  
   The portal allows structured Excel downloads, usually in batches of up to 300 cases.

2. **EU Open Data JSON dataset**  
   The consolidated dataset is available here: https://data.europa.eu/data/datasets/18489cb7-bce7-4d44-a138-795b390d2109~~1?locale=en  
   The downloaded JSON file is expected to be named `case-data-AT.json`.

The JSON contains more nested detail than the portal Excel export. The goal of this notebook is therefore to convert the JSON into two usable spreadsheet views:

- **View 1: case-by-case view** — one row per case, similar in spirit to the Commission Excel export, but with richer summaries. Core metadata fields are separate columns. Case-level press releases and Official Journal publications are grouped into `Other case related information`, timeline events are kept in their own column, and case attachments are summarised separately.
- **View 2: document-by-document view** — one row per URL-bearing case-related document, including decisions, press releases, Official Journal publications, and case attachments. Timeline events are excluded from this view because they generally do not contain URLs.

The second view is intended as the basis for later manual classification of relevant or potentially **contestable acts**. This notebook includes a rule-based `Relevant` flag for the narrow Commission-act scope used for downloads.


**v14 patches:** consolidated decision manifest (`download_com_antitrust_decisions.csv`), tertiary case-level OJ fallback for no-URL relevant rows only, and decision/case validation based on the consolidated case+date decision layer.

**v14 patch:** Commission API date strings are parsed as `yyyy-mm-dd` and normalized to `dd/mm/yyyy` across case dates, decision dates, attachment dates, OJ publication dates, and press-release dates. Sorting uses the parsed dates rather than ambiguous locale parsing.


## 1. Project paths, logging, and JSON load

This notebook expects the project layout you described: `case-data-AT.json` in `eccjeu/input/`, the notebook in `eccjeu/code/notebooks/`, outputs in `eccjeu/output/com_antitrust/`, and logs in `eccjeu/logs/com_antitrust/`.


In [1]:
import json
import logging
import os
import re
import time
import hashlib
import mimetypes
from urllib.parse import urlparse, unquote

from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any

import pandas as pd
import requests
from tqdm.auto import tqdm

# -----------------------------------------------------------------------------
# Project paths
# -----------------------------------------------------------------------------
# Expected layout:
# eccjeu/
#   code/notebooks/          <- this notebook
#   input/case-data-AT.json  <- Commission AT JSON
#   output/com_antitrust/    <- all CSV/XLSX outputs
#   logs/com_antitrust/      <- notebook logs

NOTEBOOK_DIR = Path.cwd().resolve()

# Robustly infer project root when running from eccjeu/code/notebooks.
if NOTEBOOK_DIR.name == "notebooks" and NOTEBOOK_DIR.parent.name == "code":
    PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
else:
    # Fallback for running the notebook from another working directory.
    # Change this manually if needed.
    PROJECT_ROOT = NOTEBOOK_DIR

CODE_DIR = PROJECT_ROOT / "code"
NOTEBOOKS_DIR = CODE_DIR / "notebooks"
INPUT_DIR = PROJECT_ROOT / "input"
OUTPUT_DIR = PROJECT_ROOT / "output"
LOGS_DIR = PROJECT_ROOT / "logs"
DATA_DIR = PROJECT_ROOT / "data"

COM_ANTITRUST_OUTPUT_DIR = OUTPUT_DIR / "com_antitrust"
COM_ANTITRUST_LOGS_DIR = LOGS_DIR / "com_antitrust"
COM_ANTITRUST_DATA_DIR = DATA_DIR / "raw" / "com_antitrust"

COM_ANTITRUST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COM_ANTITRUST_LOGS_DIR.mkdir(parents=True, exist_ok=True)
COM_ANTITRUST_DATA_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = INPUT_DIR / "case-data-AT.json"

# -----------------------------------------------------------------------------
# Logging
# -----------------------------------------------------------------------------
log_file = COM_ANTITRUST_LOGS_DIR / f"com_at_manifest_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logger = logging.getLogger("com_at_manifest")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler(log_file, encoding="utf-8")
file_handler.setLevel(logging.INFO)
file_handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
logger.addHandler(file_handler)
logger.propagate = False

print("Project root:", PROJECT_ROOT)
print("Input JSON:", JSON_PATH)
print("Output directory:", COM_ANTITRUST_OUTPUT_DIR)
print("Download data directory:", COM_ANTITRUST_DATA_DIR)
print("Log file:", log_file)

# -----------------------------------------------------------------------------
# Load the Commission JSON file
# -----------------------------------------------------------------------------
if not JSON_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {JSON_PATH}. Expected case-data-AT.json in eccjeu/input/."
    )

with JSON_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

logger.info("Loaded %s cases from %s", len(data), JSON_PATH)
print(f"Loaded {len(data)} cases from JSON")


Project root: /home/edik/projects/eccjeu
Input JSON: /home/edik/projects/eccjeu/input/case-data-AT.json
Output directory: /home/edik/projects/eccjeu/output/com_antitrust
Download data directory: /home/edik/projects/eccjeu/data/raw/com_antitrust
Log file: /home/edik/projects/eccjeu/logs/com_antitrust/com_at_manifest_20260706_214754.log
Loaded 751 cases from JSON


## 2. Helper functions

The Commission JSON is semi-structured: many fields are stored as lists, JSON-encoded strings, or nested `items` blocks. The helpers below flatten these structures, construct readable summaries, and build EUR-Lex URLs for Official Journal references.

The EUR-Lex URL logic deliberately avoids treating references such as `C:2008:021:5` as CELEX identifiers. Those are Official Journal references, not always direct CELEX numbers.

In [2]:
# Utility functions for parsing and summarizing

def parse_json_list(items: List[str]) -> List[Dict[str, Any]]:
    # Parse list of JSON-encoded strings. If an object contains "items", flatten them.
    parsed = []
    for item in items or []:
        try:
            obj = json.loads(item)
            if isinstance(obj, dict) and "items" in obj:
                parsed.extend(obj["items"])
            else:
                parsed.append(obj)
        except Exception:
            parsed.append(item)
    return parsed


def parse_code_label_list(raw_list: List[str]) -> List[str]:
    # Parse list of JSON strings with "code" and "label" into labels.
    labels = []
    for item in raw_list or []:
        try:
            obj = json.loads(item)
            label = obj.get("label")
            if label:
                labels.append(label)
        except Exception:
            if item:
                labels.append(str(item))
    return labels


def first_or_blank(values):
    # Return the first value in a list field, or an empty string.
    if isinstance(values, list):
        return values[0] if values else ""
    return values if values is not None else ""


def join_values(values, sep="; "):
    # Join list values safely for spreadsheet cells.
    if not values:
        return ""
    if not isinstance(values, list):
        return str(values)
    return sep.join(str(v) for v in values if v is not None and str(v) != "")


# --- Date helpers -------------------------------------------------------------
# The Commission API stores dates as ISO strings: yyyy-mm-dd. For manual review
# and Excel output, normalize those values to dd/mm/yyyy everywhere.

def _date_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " " )).strip()


def parse_commission_date(value):
    """Parse a Commission date as yyyy-mm-dd, with dd/mm/yyyy fallback for already-normalized values."""
    text = _date_text(value)
    if not text:
        return pd.NaT

    # Primary expected API format. This avoids month/day ambiguity.
    dt = pd.to_datetime(text, format="%Y-%m-%d", errors="coerce")
    if not pd.isna(dt):
        return dt

    # Fallback for values already normalized by an earlier run or manual edit.
    dt = pd.to_datetime(text, format="%d/%m/%Y", errors="coerce")
    if not pd.isna(dt):
        return dt

    return pd.NaT


def normalize_date(value):
    """Normalize yyyy-mm-dd Commission dates to dd/mm/yyyy. Preserve unparseable values."""
    text = _date_text(value)
    if not text:
        return ""
    dt = parse_commission_date(text)
    if pd.isna(dt):
        return text
    return dt.strftime("%d/%m/%Y")


def normalize_date_list(values):
    if not values:
        return []
    if not isinstance(values, list):
        values = [values]
    return [normalized for normalized in (normalize_date(v) for v in values) if normalized]


def join_dates(values, sep="; "):
    return sep.join(normalize_date_list(values))


def date_sort_series(series):
    return series.map(parse_commission_date)


# --- EUR-Lex URL helpers -----------------------------------------------------
# Commission case JSON references such as "L:2001:166:1" or "C:2008:021:5"
# are Official Journal references: OJ series, year, issue, page. They are not
# always CELEX identifiers.
#
# Preferred generic EUR-Lex OJ URL format used here:
#   https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.L_.2001.166.01.0001.01.ENG&toc=OJ:L:2001:166:TOC
#
# For references that are incomplete or malformed, the helper returns the OJ
# issue table of contents when possible, or an empty string.

def _as_first(value, default=""):
    # Return the first item if value is a list; otherwise return value or default.
    if isinstance(value, list):
        return value[0] if value else default
    return value if value is not None else default


def parse_oj_reference(ref: str) -> Dict[str, Any]:
    # Parse OJ references like C:2008:021:5 or L:2001:166:1.
    # Missing/non-numeric issue or page values are represented as None instead
    # of raising an exception.
    if not ref:
        return {}

    parts = [p.strip() for p in str(ref).strip().split(":")]
    if len(parts) < 3:
        return {"raw": str(ref).strip()}

    series = parts[0].upper() if parts[0] else ""
    year = parts[1] if len(parts) > 1 else ""
    issue_raw = parts[2] if len(parts) > 2 else ""
    page_raw = parts[3] if len(parts) > 3 else ""

    issue = int(issue_raw) if str(issue_raw).isdigit() else None
    page = int(page_raw) if str(page_raw).isdigit() else None

    out = {
        "raw": str(ref).strip(),
        "series": series,
        "year": year,
        "issue": issue,
        "page": page,
        "issue_raw": issue_raw,
        "page_raw": page_raw,
    }

    if series and year and issue is not None:
        out["toc_ref"] = f"OJ:{series}:{year}:{issue:03d}:TOC"

    if series and year and issue is not None and page is not None:
        out["normalized"] = f"{series}:{year}:{issue:03d}:{page}"

    return out


def normalise_oj_reference(ref: str) -> str:
    parsed = parse_oj_reference(ref)
    return parsed.get("normalized") or parsed.get("raw", "")


def eurlex_oj_toc_url(ref: str, lang: str = "EN") -> str:
    parsed = parse_oj_reference(ref)
    toc_ref = parsed.get("toc_ref")
    if not toc_ref:
        return ""
    return f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/?uri={toc_ref}"


def eurlex_oj_uriserv_url(ref: str, lang: str = "EN") -> str:
    parsed = parse_oj_reference(ref)
    series = parsed.get("series")
    year = parsed.get("year")
    issue = parsed.get("issue")
    page = parsed.get("page")

    if not (series and year and issue is not None and page is not None):
        return eurlex_oj_toc_url(ref, lang=lang)

    lang = (lang or "EN").upper()
    lang_suffix_map = {"EN": "ENG", "DE": "DEU", "FR": "FRA", "IT": "ITA", "ES": "SPA", "NL": "NLD"}
    lang_suffix = lang_suffix_map.get(lang, lang)

    uri = f"uriserv:OJ.{series}_.{year}.{issue:03d}.01.{page:04d}.01.{lang_suffix}"
    toc = f"OJ:{series}:{year}:{issue:03d}:TOC"
    return f"https://eur-lex.europa.eu/legal-content/{lang}/TXT/?uri={uri}&toc={toc}"


def eurlex_url_from_oj_item(item: Dict[str, Any], lang: str = "EN") -> str:
    # Return the best available EUR-Lex URL for a case OJ publication item.
    if not isinstance(item, dict):
        return ""

    for key in ("url", "uri", "documentUrl", "eurlexUrl", "officialJournalUrl", "link"):
        value = _as_first(item.get(key), "")
        if isinstance(value, str) and value.startswith("http"):
            return value

    ref = _as_first(item.get("reference"), "")
    return eurlex_oj_uriserv_url(ref, lang=lang)


# --- Item-specific summary helpers ------------------------------------------

def get_oj_label(item):
    # Preserve the most specific OJ label available.
    if not isinstance(item, dict):
        return "OJEU publication"
    return item.get("webDescription", "") or item.get("webDescriptionLabel", "") or "OJEU publication"


def get_view2_oj_label(item):
    # In View 2, keep the broad label "OJEU publication" and append a more
    # specific label when the JSON provides one.
    specific = get_oj_label(item)
    if not specific or specific == "OJEU publication":
        return "OJEU publication"
    return f"OJEU publication: {specific}"


def summarize_press_release_items(raw_list: List[str]) -> List[str]:
    items = parse_json_list(raw_list)
    summaries = []

    for item in items:
        if not isinstance(item, dict) or not item:
            continue

        ref = item.get("reference", "")
        desc = item.get("webDescription", "") or item.get("webDescriptionLabel", "")
        date = item.get("publicationDate", "") or item.get("publishedDate", "")
        url = f"https://ec.europa.eu/commission/presscorner/detail/en/{ref}" if ref else ""

        parts = []
        if ref:
            parts.append(f"Reference: {ref}")
        if desc:
            parts.append(f"Description: {desc}")
        if date:
            parts.append(f"Date: {normalize_date(date)}")
        if url:
            parts.append(f"URL: {url}")

        if parts:
            summaries.append("; ".join(parts))

    return summaries


def summarize_oj_items(raw_list: List[str]) -> List[str]:
    items = parse_json_list(raw_list)
    summaries = []

    for item in items:
        if not isinstance(item, dict) or not item:
            continue

        ref = item.get("reference", "")
        label = get_oj_label(item)
        date = item.get("publishedDate", "")
        url = eurlex_url_from_oj_item(item)

        parts = []
        if ref:
            parts.append(f"Reference: {ref}")
        if label:
            parts.append(f"Label: {label}")
        if date:
            parts.append(f"Date: {normalize_date(date)}")
        if url:
            parts.append(f"URL: {url}")

        if parts:
            summaries.append("; ".join(parts))

    return summaries


def summarize_case_related_information(meta: Dict[str, Any]) -> str:
    # Case-level Official Journal publications and press releases only.
    # Timeline events are intentionally not included here because they generally
    # do not have URLs. OJ and press-release blocks are separated by a blank line
    # for readability in Excel.
    blocks = []

    oj_lines = []
    oj_items = summarize_oj_items(meta.get("caseOfficialJournalPublications", []))
    oj_dates = join_dates(meta.get("caseOfficialJournalPublicationsPublishedDates", []))
    if oj_items or oj_dates:
        oj_lines.append("caseOfficialJournalPublications:")
        for item in oj_items:
            oj_lines.append(f"  - {item}")
        if oj_dates:
            oj_lines.append(f"caseOfficialJournalPublicationsPublishedDates: {oj_dates}")
        blocks.append("\n".join(oj_lines))

    pr_lines = []
    pr_items = summarize_press_release_items(meta.get("casePressReleases", []))
    pr_dates = join_dates(meta.get("casePressReleasesPublicationDates", []))
    if pr_items or pr_dates:
        pr_lines.append("casePressReleases:")
        for item in pr_items:
            pr_lines.append(f"  - {item}")
        if pr_dates:
            pr_lines.append(f"casePressReleasesPublicationDates: {pr_dates}")
        blocks.append("\n".join(pr_lines))

    return "\n\n".join(blocks)

def summarize_attachments(att_list: List[Dict[str, Any]]) -> List[str]:
    # Summarise attachment metadata into readable multi-line field/value blocks.
    # Each returned string represents one attachment item. Metadata type is
    # intentionally omitted because it is not useful for manual review.
    summaries = []

    for att in att_list or []:
        attmeta = att.get("metadata", {}) if isinstance(att, dict) else {}

        cat = parse_code_label_list(attmeta.get("attachmentCategory", []))
        category = cat[0] if cat else "Attachment"

        language = first_or_blank(attmeta.get("attachmentLanguage", [])) or first_or_blank(attmeta.get("language", []))
        document_date = normalize_date(first_or_blank(attmeta.get("attachmentDocumentDate", [])))
        publication_business_date = normalize_date(first_or_blank(attmeta.get("attachmentPublicationBusinessDate", [])))
        publication_description = first_or_blank(attmeta.get("attachmentPublicationDescription", []))
        sent_date = normalize_date(first_or_blank(attmeta.get("attachmentSentDate", [])))
        link = first_or_blank(attmeta.get("attachmentLink", []))
        id_sequence = first_or_blank(attmeta.get("attachmentIdSequence", []))
        metadata_reference = first_or_blank(attmeta.get("metadataReference", []))

        parts = []
        if category:
            parts.append(f"Category: {category}")
        if language:
            parts.append(f"Language: {language}")
        if document_date:
            parts.append(f"Document Date: {document_date}")
        if publication_business_date:
            parts.append(f"Publication Business Date: {publication_business_date}")
        if publication_description:
            parts.append(f"Publication Description: {publication_description}")
        if sent_date:
            parts.append(f"Sent Date: {sent_date}")
        if id_sequence:
            parts.append(f"Attachment ID Sequence: {id_sequence}")
        if metadata_reference:
            parts.append(f"Metadata Reference: {metadata_reference}")
        if link:
            parts.append(f"URL: {link}")

        if parts:
            summaries.append("\n".join(parts))

    return summaries


def attachment_has_url(att: Dict[str, Any]) -> bool:
    if not isinstance(att, dict):
        return False
    attmeta = att.get("metadata", {})
    return bool(first_or_blank(attmeta.get("attachmentLink", [])))

def extract_attachment_links(att_list: List[Dict[str, Any]]) -> List[str]:
    # Extract unique attachmentLink URLs from case or decision attachments.
    links = []
    seen = set()
    for att in att_list or []:
        if not isinstance(att, dict):
            continue
        attmeta = att.get("metadata", {})
        link = first_or_blank(attmeta.get("attachmentLink", []))
        if link and link not in seen:
            seen.add(link)
            links.append(link)
    return links


def extract_oj_links(raw_list: List[str], lang: str = "EN") -> List[str]:
    # Extract unique EUR-Lex / Official Journal links from OJ publication items.
    # This is used separately from attachmentLink because some decisions have
    # no decision attachment, but do have decisionOfficialJournalPublications.
    links = []
    seen = set()
    for item in parse_json_list(raw_list or []):
        if not isinstance(item, dict):
            continue
        link = eurlex_url_from_oj_item(item, lang=lang)
        if link and link not in seen:
            seen.add(link)
            links.append(link)
    return links


In [3]:
# Quick date-normalization sanity checks for v13
assert normalize_date("2001-06-15") == "15/06/2001"
assert normalize_date("1995-10-20") == "20/10/1995"
assert normalize_date("15/06/2001") == "15/06/2001"
assert join_dates(["2001-06-15", "1995-10-20"]) == "15/06/2001; 20/10/1995"
print("Date normalization checks passed: yyyy-mm-dd -> dd/mm/yyyy")


Date normalization checks passed: yyyy-mm-dd -> dd/mm/yyyy


## 3. Quick EUR-Lex URL sanity check

This small check verifies that the OJ URL helper handles normal and incomplete references without crashing.

In [4]:
# Quick check for OJ URL generation
test_refs = [
    {"reference": "C:2008:021:5", "webDescription": "OJOpinionAC"},
    {"reference": "C:2008:021:6", "webDescription": "OJOpinionAC"},
    {"reference": "C:2008:021:7", "webDescription": "OJOpinionAC"},
    {"reference": "L:2001:166:1", "webDescription": "OJSummaryDecision"},
    {"reference": "C:2008:021:", "webDescription": "Incomplete reference should not crash"},
]
for item in test_refs:
    print(item["reference"], "->", eurlex_url_from_oj_item(item))


C:2008:021:5 -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.C_.2008.021.01.0005.01.ENG&toc=OJ:C:2008:021:TOC
C:2008:021:6 -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.C_.2008.021.01.0006.01.ENG&toc=OJ:C:2008:021:TOC
C:2008:021:7 -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.C_.2008.021.01.0007.01.ENG&toc=OJ:C:2008:021:TOC
L:2001:166:1 -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.L_.2001.166.01.0001.01.ENG&toc=OJ:L:2001:166:TOC
C:2008:021: -> https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=OJ:C:2008:021:TOC


## 4. Build the two manifest views

### View 1 — case-level manifest

The first view contains one row per case. Most core metadata fields are kept as separate columns. Blank metadata fields that are not useful in the current extract, such as court cases, case links, and case external links, are intentionally omitted.

Case-level Official Journal publications and press releases are grouped into **Other case related information**, with a blank line between the OJ block and press-release block. The values are written in a field/value style that mirrors the decision summaries, for example:

```text
caseOfficialJournalPublications:
  - Reference: C:...
caseOfficialJournalPublicationsPublishedDates: ...

casePressReleases:
  - Reference: IP_...
casePressReleasesPublicationDates: ...
```

Timeline events are intentionally omitted because they are almost empty in the current extract. Case attachments are kept in their own **Case attachments** column and are formatted as readable multi-line field/value blocks, separated by blank lines between attachment items. Metadata type is intentionally omitted.

View 2 is limited to decision rows and case-attachment rows only; case-level OJ publications and press releases from **Other case related information** are not exploded into View 2.

### View 2 — document-level manifest

The second view contains one row per decision or URL-bearing case attachment. It is sorted by `Initiation of proceedings` first and `Case number` second, includes a sequential identifier (`COM1`, `COM2`, etc.), and has an `Item link` column followed by an `Item OJ link` column. `Item OJ link` is populated from `decisionOfficialJournalPublications` for decision rows, so decisions without attachment links can still point to their Official Journal/EUR-Lex reference. For case attachments, `Item date` uses `attachmentDocumentDate`; `attachmentPublicationBusinessDate` remains visible in `Item details`.

This version includes `Relevant`, using a positive inclusion rule: only prohibition, commitment, settlement, exemption/negative-clearance, interim-measures, fines/penalty-payment, readoption/amending, and Article 106 state-measure decisions are relevant. Rejection-of-complaint decisions are deliberately not relevant.


In [5]:
def summarize_decision(decision: Dict[str, Any]) -> str:
    # Construct a detailed summary string for a single decision.
    if not decision:
        return ""

    dmeta = decision.get("metadata", {})
    lines = []

    adoption = dmeta.get("decisionAdoptionDate", [])
    if adoption:
        lines.append(f"Adoption date: {normalize_date(adoption[0])}")

    number = dmeta.get("decisionNumber", [])
    if number:
        lines.append(f"Decision number: {number[0]}")

    dtypes = parse_code_label_list(dmeta.get("decisionTypes", []))
    if dtypes:
        lines.append(f"Decision type: {dtypes[0]}")

    oj_list = summarize_oj_items(dmeta.get("decisionOfficialJournalPublications", []))
    oj_dates = join_dates(dmeta.get("decisionOfficialJournalPublicationsPublishedDates", []))
    if oj_list or oj_dates:
        lines.append("decisionOfficialJournalPublications:")
        for item in oj_list:
            lines.append(f"  - {item}")
        if oj_dates:
            lines.append(f"decisionOfficialJournalPublicationsPublishedDates: {oj_dates}")

    pr_list = summarize_press_release_items(dmeta.get("decisionPressReleases", []))
    pr_dates = join_dates(dmeta.get("decisionPressReleasesPublicationDates", []))
    if pr_list or pr_dates:
        lines.append("decisionPressReleases:")
        for item in pr_list:
            lines.append(f"  - {item}")
        if pr_dates:
            lines.append(f"decisionPressReleasesPublicationDates: {pr_dates}")

    ext_summaries = summarize_external_links(dmeta.get("decisionExternalLinks", [])) if "summarize_external_links" in globals() else []
    if ext_summaries:
        lines.append("decisionExternalLinks:")
        for item in ext_summaries:
            lines.append(f"  - {item}")

    attach_summaries = summarize_attachments(decision.get("decisionAttachments", []))
    if attach_summaries:
        lines.append("decisionAttachments:")
        lines.append("\n\n".join(attach_summaries))

    return "\n".join(lines)


def build_case_base_row(case_id: str, case: Dict[str, Any]) -> Dict[str, Any]:
    # Shared case metadata used by both views.
    #
    # Deliberately excluded here:
    # - caseCourtCases, because it is blank in this dataset extract.
    # - caseLinks and caseExternalLinks, because they are blank in this dataset extract.
    # - casePressReleases and caseOfficialJournalPublications, because these are
    #   grouped into "Other case related information" in View 1 only.
    # - caseTimelineEvents, because these are almost empty in this dataset and
    #   not useful for review.
    # - metadataType, because it is not useful for manual review.
    meta = case.get("metadata", {})
    row = {
        "Case number": first_or_blank(meta.get("caseNumber", [])) or case_id,
        "Case title": first_or_blank(meta.get("caseTitle", [])),
        "Case instrument": join_values(meta.get("caseInstrument", [])),
        "Case type": join_values(meta.get("caseType", [])),
        "Antitrust / Cartels": first_or_blank(meta.get("caseCartel", [])),
        "Companies": join_values(meta.get("caseCompanies", [])),
        "DG": join_values(meta.get("caseDg", [])),
        "Case number part": join_values(meta.get("caseNumberPart", [])),
        "Last decision date": normalize_date(first_or_blank(meta.get("caseLastDecisionDate", []))),
        "Initiation of proceedings": normalize_date(first_or_blank(meta.get("caseInitiationDate", []))),
        "Economic activities": join_values(parse_code_label_list(meta.get("caseSectors", []))),
        "Legal basis": join_values(parse_code_label_list(meta.get("caseLegalBasis", []))),
        "Case language": join_values(meta.get("language", [])),
        "Metadata reference": join_values(meta.get("metadataReference", [])),
    }
    return row


def build_view1(data: Dict[str, Any]) -> pd.DataFrame:
    # Build one row per case.
    rows = []

    for case_id, case in data.items():
        meta = case.get("metadata", {})
        row = build_case_base_row(case_id, case)

        decision_summaries = []
        for decision in case.get("decisions", []):
            summary = summarize_decision(decision)
            if summary:
                decision_summaries.append(summary)
        row["Decisions"] = "\n\n".join(decision_summaries) if decision_summaries else None

        attachment_summaries = summarize_attachments(case.get("caseAttachments", []))
        row["Case attachments"] = "\n\n".join(attachment_summaries) if attachment_summaries else None

        # Case-level press releases and OJ publications are grouped here.
        # They are related documents/publications rather than core metadata.
        other_info = summarize_case_related_information(meta)
        row["Other case related information"] = other_info if other_info else None

        rows.append(row)

    return pd.DataFrame(rows)



# -----------------------------------------------------------------------------
# Relevance rules for View 2
# -----------------------------------------------------------------------------
# The Relevant flag is intentionally a POSITIVE inclusion rule. A document is
# relevant only when its item label falls into the narrow Commission-act scope:
#
# - Prohibition Decision
# - Commitment Decision
# - Settlement Decision
# - Exemption / Negative Clearance
# - Interim Measures
# - Fines / Penalty Payment
# - Readoption / Amending Decision
# - State Measure Decision, but only where the legal basis contains Article 106
#
# Rejection-of-complaint decisions are deliberately excluded.

RELEVANT_EXACT_ITEM_LABELS = {
    # Prohibition
    "Prohibition Decision",
    "Prohibition Decision (Art. 7) (Art. 101 & 102 ex 81 & 82)",

    # Commitments / settlements
    "Commitment Decision",
    "Commitments decision (Art. 9)",
    "Settlement Decision",

    # Exemption / negative clearance
    "Exemption without condition (Reg 17/62)",
    "Old milestones - Exemption with condition decision",
    "Negative Clearance Decision (Reg 17/62)",
    "Old milestones - Negative clearance decision",

    # Interim measures
    "Interim Measures Decision",
    "Interim measures Decision (Art. 8)",

    # Fines / penalty payments
    "Decision imposing fines",
    "Fines Decision (Art. 23)",
    "Periodic Penalty Payment Decision (Art. 24)",
    "Penalty Payments to comply with Commission Decisions",
    "LPP Decision",

    # Readoption / amendment
    "Amending Decision",
    "Readoption Decision",
    "Re-adoption Decision",
}

RELEVANT_LABEL_PATTERNS = [
    # Defensive patterns for future extracts / label wording variants.
    r"\bprohibition\b.*\bdecision\b",
    r"\bcommitment(s)?\b.*\bdecision\b",
    r"\bsettlement\b.*\bdecision\b",
    r"\bexemption\b.*\bdecision\b",
    r"\bnegative\s+clearance\b.*\bdecision\b",
    r"\binterim\s+measures?\b.*\bdecision\b",
    r"\bfines?\b.*\bdecision\b",
    r"\bpenalty\s+payment\b.*\bdecision\b",
    r"\breadopt(?:ion|ed)?\b.*\bdecision\b",
    r"\bre-adopt(?:ion|ed)?\b.*\bdecision\b",
    r"\bamending\b.*\bdecision\b",
]

ALWAYS_EXCLUDE_ITEM_LABELS = {
    "Rejection of Complaint Decision",
    "Closure of Proceedings",
    "Closure of proceedings",
    "Administrative Closure",
    "Summary Decision",
    "Sector Inquiry - Final Report",
    "Proposed Commitments",
    "Commitments - Final",  # final commitment text, not the Commission commitment decision row
    "Cooperation Decision",
    "Trustee Approval Decision",
    "Trustee details",
    "Market test notice (Art. 27(4))",
    "Press Release / Memo",
    "Advisory Committee-Opinion of the Member States",
    "Hearing Officer - Final Report",
    "Initiation of Proceedings",
    "Initiation of proceedings Notice (Art. 11(6))",
    "Informal Guidance Letter",
    "Oral Hearing",
    "Other publication",
    "Provisional non-confidential version of the decision",
    "Report",
    "Response(s) to the Statement of Objections",
}


def is_article_106_legal_basis(legal_basis: Any) -> bool:
    text = str(legal_basis or "").lower()
    return bool(re.search(r"\bart\.?\s*106\b|\barticle\s*106\b", text))


def is_view2_relevant_item_label(item_label: Any, legal_basis: Any = "") -> bool:
    """Return True only for the agreed narrow Commission-act scope."""
    label = str(item_label or "").strip()
    if not label:
        return False

    if label in ALWAYS_EXCLUDE_ITEM_LABELS:
        return False

    # Article 106 state-measure cases are relevant only where the legal basis
    # confirms Article 106. This prevents non-106 state-measure-like labels from
    # leaking in if the source data changes.
    if label == "State Measure Decision":
        return is_article_106_legal_basis(legal_basis)

    if label in RELEVANT_EXACT_ITEM_LABELS:
        return True

    label_lower = label.lower()
    return any(re.search(pattern, label_lower) for pattern in RELEVANT_LABEL_PATTERNS)



def get_case_level_oj_links(meta: Dict[str, Any], lang: str = "EN") -> List[str]:
    """Extract case-level OJ URLs from the Commission case metadata.

    These are used only as tertiary fallback URLs when a relevant decision or
    case attachment has neither a main document URL nor its own decision-level
    OJ URL.
    """
    return extract_oj_links(meta.get("caseOfficialJournalPublications", []), lang=lang)


def classify_url_availability(item_links: List[str], oj_links: List[str], tertiary_oj_links: List[str]) -> str:
    """Classify which URL layer is available for a document/decision row."""
    if item_links:
        return "main_url_available"
    if oj_links:
        return "secondary_oj_available"
    if tertiary_oj_links:
        return "tertiary_oj_found"
    return "no_url_available"


def build_view2(data: Dict[str, Any]) -> pd.DataFrame:
    # Build one row per review document / item.
    #
    # This view is intentionally limited to core review documents:
    # - decision rows
    # - case-attachment rows
    #
    # NEW in v6:
    # - Case attachments are kept even if they do not have their own URL, so
    #   relevance and no-url gaps remain visible.
    # - If a relevant decision/case attachment has no main URL and no own OJ URL,
    #   the notebook uses case-level OJ references as a tertiary fallback.
    #   This is deliberately only done when no row-level URL exists.
    rows = []

    for case_id, case in data.items():
        meta = case.get("metadata", {})
        base_row = build_case_base_row(case_id, case)
        case_level_oj_links = get_case_level_oj_links(meta)

        # Decisions. The Item link column captures the decisionAttachments
        # attachmentLink URL(s), where present. Item OJ link captures OJ links
        # listed directly on the decision. Item tertiary OJ link captures
        # case-level OJ links only if no row-level URL is available.
        for decision in case.get("decisions", []):
            dmeta = decision.get("metadata", {})
            adoption_date = normalize_date(first_or_blank(dmeta.get("decisionAdoptionDate", [])))
            dtypes = parse_code_label_list(dmeta.get("decisionTypes", []))
            dlabel = dtypes[0] if dtypes else None

            item_links = extract_attachment_links(decision.get("decisionAttachments", []))
            oj_links = extract_oj_links(dmeta.get("decisionOfficialJournalPublications", []))
            relevant = is_view2_relevant_item_label(dlabel, base_row.get("Legal basis", ""))

            # Tertiary OJ fallback: only relevant rows, and only if the decision
            # has neither a main URL nor its own decision-level OJ URL.
            tertiary_oj_links = case_level_oj_links if relevant and not item_links and not oj_links else []

            row = base_row.copy()
            row["Item type"] = "Decision"
            row["Item date"] = adoption_date or None
            row["Item label"] = dlabel
            row["Relevant"] = relevant
            row["Item link"] = join_values(item_links) or None
            row["Item OJ link"] = join_values(oj_links) or None
            row["Item tertiary OJ link"] = join_values(tertiary_oj_links) or None
            row["URL availability"] = classify_url_availability(item_links, oj_links, tertiary_oj_links)
            row["Item details"] = summarize_decision(decision)
            rows.append(row)

        # Case attachments. In v6, keep attachments even when attachmentLink is
        # missing. This is important because relevant case attachments without
        # direct URLs may still be recoverable through case-level OJ references.
        for att in case.get("caseAttachments", []):
            if not isinstance(att, dict):
                continue

            attmeta = att.get("metadata", {}) if isinstance(att, dict) else {}
            cat = parse_code_label_list(attmeta.get("attachmentCategory", []))
            cat_label = cat[0] if cat else "Attachment"
            document_date = normalize_date(first_or_blank(attmeta.get("attachmentDocumentDate", [])))
            item_link = first_or_blank(attmeta.get("attachmentLink", []))
            item_links = [item_link] if item_link else []
            oj_links = []

            details_list = summarize_attachments([att])
            details = details_list[0] if details_list else ""
            relevant = is_view2_relevant_item_label(cat_label, base_row.get("Legal basis", ""))

            # Tertiary OJ fallback: only relevant attachments, and only if the
            # attachment has no own URL.
            tertiary_oj_links = case_level_oj_links if relevant and not item_links and not oj_links else []

            row = base_row.copy()
            row["Item type"] = "Case attachment"
            row["Item date"] = document_date or None
            row["Item label"] = cat_label
            row["Relevant"] = relevant
            row["Item link"] = item_link or None
            row["Item OJ link"] = None
            row["Item tertiary OJ link"] = join_values(tertiary_oj_links) or None
            row["URL availability"] = classify_url_availability(item_links, oj_links, tertiary_oj_links)
            row["Item details"] = details
            rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Sort chronologically by initiation of proceedings first, then by case
    # number. Missing initiation dates are placed last.
    sort_date = date_sort_series(df["Initiation of proceedings"])
    item_date = date_sort_series(df["Item date"])
    df = (
        df.assign(_sort_initiation=sort_date, _sort_item_date=item_date)
          .sort_values(
              by=["_sort_initiation", "Case number", "_sort_item_date", "Item type"],
              ascending=[True, True, True, True],
              na_position="last",
              kind="mergesort",
          )
          .drop(columns=["_sort_initiation", "_sort_item_date"])
          .reset_index(drop=True)
    )
    df.insert(0, "Identifier", [f"COM{i}" for i in range(1, len(df) + 1)])

    # Keep Relevant immediately after Item label; keep URL columns together.
    cols = list(df.columns)

    if "Item label" in cols and "Relevant" in cols:
        cols.remove("Relevant")
        insert_at = cols.index("Item label") + 1
        cols.insert(insert_at, "Relevant")

    # Ordered URL-related columns.
    url_cols = ["Item link", "Item OJ link", "Item tertiary OJ link", "URL availability"]
    for col in url_cols:
        if col in cols:
            cols.remove(col)
    if "Relevant" in cols:
        insert_at = cols.index("Relevant") + 1
    elif "Item label" in cols:
        insert_at = cols.index("Item label") + 1
    else:
        insert_at = len(cols)
    for offset, col in enumerate(url_cols):
        if col in df.columns:
            cols.insert(insert_at + offset, col)

    df = df[cols]

    return df


## 5. Export to CSV and Excel

The final cell writes all CSV and Excel outputs to `eccjeu/output/com_antitrust/` and logs the run in `eccjeu/logs/com_antitrust/`.


In [6]:
# Build the two views
view1 = build_view1(data)
view2 = build_view2(data)

logger.info("Built View 1 with shape %s", view1.shape)
logger.info("Built View 2 with shape %s", view2.shape)

# Output paths
view1_csv_path = COM_ANTITRUST_OUTPUT_DIR / "download_com_antitrust_cases.csv"
view2_csv_path = COM_ANTITRUST_OUTPUT_DIR / "download_com_antitrust_documents.csv"
excel_path = COM_ANTITRUST_OUTPUT_DIR / "download_com_antitrust.xlsx"

# Save to CSV files
view1.to_csv(view1_csv_path, index=False, encoding="utf-8-sig")
view2.to_csv(view2_csv_path, index=False, encoding="utf-8-sig")
logger.info("Saved View 1 CSV to %s", view1_csv_path)
logger.info("Saved View 2 CSV to %s", view2_csv_path)
print("Saved CSV files:")
print(" -", view1_csv_path)
print(" -", view2_csv_path)

# Save to Excel workbook with two sheets
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    view1.to_excel(writer, sheet_name="View 1 - cases", index=False)
    view2.to_excel(writer, sheet_name="View 2 - documents", index=False)

    # Improve readability of multi-line attachment/decision cells in Excel.
    from openpyxl.styles import Alignment, Font

    for sheet_name in ["View 1 - cases", "View 2 - documents"]:
        ws = writer.book[sheet_name]
        ws.freeze_panes = "A2"

        # Header formatting
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.alignment = Alignment(wrap_text=True, vertical="top")

        # Cell wrapping
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

        # Column widths
        for col_cells in ws.columns:
            header = col_cells[0].value
            if header in {"Decisions", "Case attachments", "Other case related information", "Item details", "Item link", "Item OJ link"}:
                ws.column_dimensions[col_cells[0].column_letter].width = 70
            else:
                ws.column_dimensions[col_cells[0].column_letter].width = 22

logger.info("Saved Excel workbook to %s", excel_path)
print("Excel workbook saved:", excel_path)
print("View 1 shape:", view1.shape)
print("View 2 shape:", view2.shape)


Saved CSV files:
 - /home/edik/projects/eccjeu/output/com_antitrust/download_com_antitrust_cases.csv
 - /home/edik/projects/eccjeu/output/com_antitrust/download_com_antitrust_documents.csv
Excel workbook saved: /home/edik/projects/eccjeu/output/com_antitrust/download_com_antitrust.xlsx
View 1 shape: (751, 17)
View 2 shape: (1189, 24)


In [7]:
# Relevance sanity checks
print("View 2 relevance counts:")
display(view2["Relevant"].value_counts(dropna=False))

print("Relevance by item label:")
display(
    view2.groupby(["Item label", "Relevant"], dropna=False)
         .size()
         .reset_index(name="rows")
         .sort_values(["Relevant", "Item label"], ascending=[False, True])
)

# Rejection-of-complaint decisions must never be relevant.
rejection_mask = view2["Item label"].fillna("").eq("Rejection of Complaint Decision")
assert not view2.loc[rejection_mask, "Relevant"].fillna(False).any(), "Rejection-of-complaint rows were incorrectly flagged relevant."

# State Measure Decision rows are relevant only when Article 106 is in the legal basis.
state_mask = view2["Item label"].fillna("").eq("State Measure Decision")
if state_mask.any():
    state_check = view2.loc[state_mask, ["Case number", "Item label", "Legal basis", "Relevant"]].copy()
    display(state_check)
    assert (
        state_check["Relevant"].fillna(False).astype(bool)
        == state_check["Legal basis"].fillna("").astype(str).str.contains(r"\bArt\.?\s*106\b|\bArticle\s*106\b", case=False, regex=True)
    ).all(), "State Measure relevance does not match Article 106 legal basis."


View 2 relevance counts:


Relevant
False    751
True     438
Name: count, dtype: int64

Relevance by item label:


,Item label,Relevant,rows
2,Amending Decision,True,8
5,Commitment Decision,True,65
7,Commitments decision (Art. 9),True,6
9,Decision imposing fines,True,5
10,Exemption without condition (Reg 17/62),True,21
11,Fines Decision (Art. 23),True,2
16,Interim Measures Decision,True,2
17,Interim measures Decision (Art. 8),True,2
18,LPP Decision,True,1
20,Negative Clearance Decision (Reg 17/62),True,1


,Case number,Item label,Legal basis,Relevant
45,AT.35703,State Measure Decision,Art. 106 + Art. 102,True
47,AT.35737,State Measure Decision,Art. 106 + Art. 102,True
189,AT.37133,State Measure Decision,Art. 106 + Art. 102,True
238,AT.37721,State Measure Decision,Art. 106 + Art. 102,True
403,AT.38700,State Measure Decision,Art. 106 + Art. 102,True
404,AT.38700,State Measure Decision,Art. 106 + Art. 102,True
409,AT.38700,State Measure Decision,Art. 106 + Art. 102,True
411,AT.38700,State Measure Decision,Art. 106 + Art. 102,True
414,AT.38745,State Measure Decision,Art. 106 + Art. 102,True
569,AT.39562,State Measure Decision,Art. 106 + Art. 102,True


## 5b. Consolidate document rows into decisions

A decision is treated as a unique **case number + decision/document date** pairing. This gives one decision-level row even when the Commission registry has several document rows for the same decision date. If the same case/date group contains conflicting relevant labels, the consolidated `Decision label` is set to `Unclear Decision` and the original labels remain available in `Decision labels`.

In [8]:

# -----------------------------------------------------------------------------
# Consolidated decision manifest
# -----------------------------------------------------------------------------
# A decision is defined as a unique case + date pairing. Relevant document rows
# from View 2 are consolidated into one decision row. If a case/date group
# contains conflicting labels, it is explicitly marked as "Unclear Decision".

DOWNLOAD_COM_ANTITRUST_DECISIONS_CSV = COM_ANTITRUST_OUTPUT_DIR / "download_com_antitrust_decisions.csv"



# Local helper copies make this cell order-safe.
# They are also defined later in the downloader section, but the decision
# manifest is built before the downloader configuration cell.
def clean_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " ")).strip()


def hash_text(value, n=16):
    return hashlib.sha256(clean_text(value).encode("utf-8")).hexdigest()[:n]


def normalize_decision_key_part(value):
    text = clean_text(value)
    text = re.sub(r"\s+", "", text)
    text = text.replace("/", "-")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text or "missing"


def make_decision_key(case_number, decision_date):
    """Human-readable decision key: case_no+date."""
    return f"{normalize_decision_key_part(case_number)}+{normalize_decision_key_part(decision_date)}"


def make_decision_key_reverse(case_number, decision_date):
    """Human-readable reverse decision key: date_caseno."""
    return f"{normalize_decision_key_part(decision_date)}_{normalize_decision_key_part(case_number)}"


def split_manifest_links(value):
    """Split semicolon/newline-separated URL strings and preserve first-seen order."""
    value = clean_text(value)
    if not value:
        return []
    parts = re.split(r"\s*;\s*|\s*\n+\s*", value)
    links = []
    for part in parts:
        part = clean_text(part)
        if part.startswith("http"):
            links.append(part)
    return list(dict.fromkeys(links))

def _truthy_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return s.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def _join_unique_preserve_order(values, sep="; ") -> str:
    seen = set()
    out = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text.lower() == "nan":
            continue
        for part in re.split(r"\s*;\s*|\n+", text):
            part = part.strip()
            if part and part not in seen:
                seen.add(part)
                out.append(part)
    return sep.join(out)


def _count_links(values) -> int:
    return len(split_manifest_links(_join_unique_preserve_order(values)))


def _decision_label_from_group(labels) -> str:
    unique_labels = []
    seen = set()
    for value in labels:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text not in seen:
            seen.add(text)
            unique_labels.append(text)

    if len(unique_labels) == 0:
        return "Unclear Decision"
    if len(unique_labels) == 1:
        return unique_labels[0]
    return "Unclear Decision"


def _decision_url_availability(row: pd.Series) -> str:
    if row["main_url_count"] > 0:
        return "main_url_available"
    if row["secondary_oj_url_count"] > 0:
        return "secondary_oj_available"
    if row["tertiary_oj_url_count"] > 0:
        return "tertiary_oj_found"
    return "no_url_available"


def build_com_antitrust_decisions(view2_df: pd.DataFrame) -> pd.DataFrame:
    required = {
        "Identifier", "Case number", "Case title", "Item type", "Item date",
        "Item label", "Relevant", "Item link", "Item OJ link", "Item details"
    }
    missing = sorted(required - set(view2_df.columns))
    if missing:
        raise ValueError(f"view2 missing required columns for decision consolidation: {missing}")

    df = view2_df.copy()
    df["Relevant_bool"] = _truthy_series(df["Relevant"])
    relevant = df[df["Relevant_bool"]].copy()

    if relevant.empty:
        return pd.DataFrame()

    if "Item tertiary OJ link" not in relevant.columns:
        relevant["Item tertiary OJ link"] = ""

    relevant["decision_group_case"] = relevant["Case number"].fillna("").astype(str).str.strip()
    relevant["decision_group_date"] = relevant["Item date"].fillna("").astype(str).str.strip()

    group_cols = ["decision_group_case", "decision_group_date"]

    agg = (
        relevant.groupby(group_cols, dropna=False)
                .agg(
                    case_number=("Case number", "first"),
                    case_title=("Case title", "first"),
                    decision_date=("Item date", "first"),
                    decision_label=("Item label", _decision_label_from_group),
                    decision_labels=("Item label", _join_unique_preserve_order),
                    item_types=("Item type", _join_unique_preserve_order),
                    row_identifiers=("Identifier", _join_unique_preserve_order),
                    relevant_document_rows=("Identifier", "size"),
                    main_urls=("Item link", _join_unique_preserve_order),
                    secondary_oj_urls=("Item OJ link", _join_unique_preserve_order),
                    tertiary_oj_urls=("Item tertiary OJ link", _join_unique_preserve_order),
                    url_availability_rows=("URL availability", _join_unique_preserve_order),
                    legal_basis=("Legal basis", _join_unique_preserve_order),
                    initiation_of_proceedings=("Initiation of proceedings", "first"),
                    case_instrument=("Case instrument", _join_unique_preserve_order),
                    case_type=("Case type", _join_unique_preserve_order),
                    antitrust_cartels=("Antitrust / Cartels", _join_unique_preserve_order),
                    item_details=("Item details", _join_unique_preserve_order),
                )
                .reset_index(drop=True)
    )

    agg["main_url_count"] = relevant.groupby(group_cols, dropna=False)["Item link"].apply(_count_links).to_numpy()
    agg["secondary_oj_url_count"] = relevant.groupby(group_cols, dropna=False)["Item OJ link"].apply(_count_links).to_numpy()
    agg["tertiary_oj_url_count"] = relevant.groupby(group_cols, dropna=False)["Item tertiary OJ link"].apply(_count_links).to_numpy()
    agg["url_availability"] = agg.apply(_decision_url_availability, axis=1)
    agg["is_unclear_decision"] = agg["decision_label"].eq("Unclear Decision")

    # Stable, human-readable decision identifiers based on the user's definition:
    # a decision is a unique case + date pairing.
    agg.insert(
        0,
        "decision_key",
        agg.apply(lambda r: make_decision_key(r["case_number"], r["decision_date"]), axis=1),
    )
    agg.insert(
        1,
        "decision_key_reverse",
        agg.apply(lambda r: make_decision_key_reverse(r["case_number"], r["decision_date"]), axis=1),
    )
    agg.insert(0, "Decision identifier", [f"COMDEC{i:05d}" for i in range(1, len(agg) + 1)])

    order_cols = [
        "Decision identifier", "decision_key", "decision_key_reverse", "case_number", "case_title",
        "decision_date", "decision_label", "is_unclear_decision",
        "decision_labels", "item_types", "relevant_document_rows",
        "url_availability", "main_url_count", "secondary_oj_url_count",
        "tertiary_oj_url_count", "main_urls", "secondary_oj_urls",
        "tertiary_oj_urls", "row_identifiers", "legal_basis",
        "initiation_of_proceedings", "case_instrument", "case_type",
        "antitrust_cartels", "item_details",
    ]
    existing = [c for c in order_cols if c in agg.columns]
    remaining = [c for c in agg.columns if c not in existing]
    return agg[existing + remaining]


decision_manifest = build_com_antitrust_decisions(view2)
decision_manifest.to_csv(DOWNLOAD_COM_ANTITRUST_DECISIONS_CSV, index=False, encoding="utf-8-sig")

print("Saved consolidated decision manifest:")
print(" -", DOWNLOAD_COM_ANTITRUST_DECISIONS_CSV)
print("Decision rows:", len(decision_manifest))
print("Unclear decisions:", int(decision_manifest.get("is_unclear_decision", pd.Series(dtype=bool)).sum()))

print("\nDecision URL availability:")
display(decision_manifest["url_availability"].value_counts(dropna=False))


Saved consolidated decision manifest:
 - /home/edik/projects/eccjeu/output/com_antitrust/download_com_antitrust_decisions.csv
Decision rows: 400
Unclear decisions: 4

Decision URL availability:


url_availability
main_url_available        290
secondary_oj_available     80
tertiary_oj_found          16
no_url_available           14
Name: count, dtype: int64

## 6. Build download manifest from View 2 and optionally download files

This section mirrors the `download_com_db_comp` workflow, but uses the COM antitrust document view as the source.

For every View 2 row, the notebook checks both `Item link` and `Item OJ link`.

- Rows with neither link are kept in the manifest as `no_download_available`.
- EUR-Lex `legal-content` URLs are attempted as originally provided first. If that fails, the downloader then tries nested `/TXT/HTML/` and `/TXT/PDF/` fallback candidates once each.
- Downloaded files are saved in `data/raw/com_antitrust/`.
- Results are written to `output/com_antitrust/download_manifest.csv` and `.parquet` when possible. After downloading, the consolidated decision manifest is rewritten with supporting-document evidence columns.


In [9]:

# -----------------------------------------------------------------------------
# COM antitrust download configuration
# -----------------------------------------------------------------------------

# Download/status mode. This is the main switch for v14.
#
# - "rebuild_jsonl": overwrite the JSONL completely, check local files, then attempt all missing files.
# - "update_jsonl": keep existing JSONL, check local files, then attempt only rows not successfully downloaded.
# - "update_csv_from_jsonl": do not download; rebuild download_manifest.csv/parquet from JSONL + local files.
DOWNLOAD_MODE = "update_jsonl"

# Backward-compatible switch: if False, no network downloads happen even in rebuild/update JSONL modes.
DOWNLOAD_FILES = True

OVERWRITE_EXISTING_FILES = False
DOWNLOAD_LIMIT = None          # e.g. 20 for testing; None = all pending/non-downloaded rows
SAVE_EVERY = 25
DOWNLOAD_SLEEP_SECONDS = (1.5, 4.0)   # moderate polite delay between attempted rows
DOWNLOAD_TIMEOUT_SECONDS = 25

# v12 retry profile: much faster than v11, but still more polite than the original fast mode.
# 202 Accepted usually means EUR-Lex accepted the request but did not return the generated
# representation immediately. Do not wait 90 seconds per row; try briefly, then move on.

# EUR-Lex sometimes returns HTTP 202 Accepted while preparing generated HTML/PDF/TXT.
# These settings make the downloader wait and retry the same candidate before moving
# on to the next fallback candidate. The waits are deliberately conservative.
EURLEX_PRE_REQUEST_SLEEP_SECONDS = 0.75
EURLEX_202_MAX_RETRIES = 4
EURLEX_202_SLEEP_SECONDS = 8.0
EURLEX_202_BACKOFF_FACTOR = 1.8
EURLEX_202_MAX_SLEEP_SECONDS = 45.0

# Slow down only for genuine blocking/throttling signals.
HTTP_403_SLEEP_SECONDS = (20.0, 45.0)
HTTP_429_SLEEP_SECONDS = (30.0, 90.0)

# Download manifest/download loop should only contain rows flagged Relevant.
# Set to False if you want a diagnostic manifest containing non-relevant rows too.
DOWNLOAD_ONLY_RELEVANT = True

HEADERS = {
    # Use a normal browser-like user agent. EUR-Lex can behave differently for
    # bare Python clients, especially when it needs to generate HTML/PDF output.
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,application/pdf;q=0.8,*/*;q=0.7",
    "Accept-Language": "en-US,en;q=0.9,de;q=0.8,fr;q=0.7",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1",
}

DOWNLOAD_MANIFEST_CSV = COM_ANTITRUST_OUTPUT_DIR / "download_manifest.csv"
DOWNLOAD_MANIFEST_PARQUET = COM_ANTITRUST_OUTPUT_DIR / "download_manifest.parquet"
DOWNLOAD_ATTEMPTS_JSONL = COM_ANTITRUST_OUTPUT_DIR / "download_attempts.jsonl"


DOWNLOAD_MANIFEST_COLUMNS = [
    "download_key",
    "identifier",
    "case_number",
    "case_title",
    "initiation_of_proceedings",
    "item_type",
    "item_date",
    "decision_key",
    "decision_key_reverse",
    "item_label",
    "relevant",
    "source_column",
    "source_url",
    "source_role",
    "download_url",
    "download_url_kind",
    "source_url_candidates_json",
    "url_candidate_order",
    "download_file_name",
    "download_path",
    "download_success",
    "download_status",
    "http_status",
    "content_type",
    "bytes",
    "error",
    "local_file_exists",
    "local_file_bytes",
]


def clean_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"\s+", " ", str(value).replace("\u00a0", " ")).strip()


def hash_text(value, n=16):
    return hashlib.sha256(clean_text(value).encode("utf-8")).hexdigest()[:n]


def normalize_decision_key_part(value):
    text = clean_text(value)
    text = re.sub(r"\s+", "", text)
    text = text.replace("/", "-")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text or "missing"


def make_decision_key(case_number, decision_date):
    """Human-readable decision key: case_no+date."""
    return f"{normalize_decision_key_part(case_number)}+{normalize_decision_key_part(decision_date)}"


def make_decision_key_reverse(case_number, decision_date):
    """Human-readable reverse decision key: date_caseno."""
    return f"{normalize_decision_key_part(decision_date)}_{normalize_decision_key_part(case_number)}"


def safe_filename(value, default="document"):
    value = clean_text(value)
    if not value:
        value = default
    value = value.replace("/", "-").replace("\\", "-")
    value = re.sub(r"\s+", "_", value)
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("._-")
    return value or default


def split_manifest_links(value):
    """Split links stored either as semicolon/newline-separated strings or as one URL."""
    value = clean_text(value)
    if not value:
        return []
    parts = re.split(r"\s*;\s*|\s*\n+\s*", value)
    links = []
    for part in parts:
        part = clean_text(part)
        if part.startswith("http"):
            links.append(part)
    # Preserve order while deduplicating.
    return list(dict.fromkeys(links))


def expand_download_url_candidates(url):
    """
    Return candidate URLs to try for a source URL.

    EUR-Lex / OJ rule:
    - Keep the original EUR-Lex legal-content URL as candidate 1.
    - If candidate 1 fails, try the nested /TXT/HTML/ form once.
    - If that fails, try the nested /TXT/PDF/ form once.
    - Do not try the fallback candidates if an earlier candidate succeeds; the
      download loop marks later candidates for the same source URL as skipped.

    Example:
        .../legal-content/EN/TXT/?uri=...
          -> .../legal-content/EN/TXT/?uri=...
          -> .../legal-content/EN/TXT/HTML/?uri=...
          -> .../legal-content/EN/TXT/PDF/?uri=...
    """
    url = clean_text(url)
    if not url:
        return []

    # Non-EUR-Lex links are kept exactly as provided.
    if "eur-lex.europa.eu/legal-content/" not in url:
        return [url]

    candidates = [url]

    html_candidate = None
    pdf_candidate = None

    if "/TXT/HTML/" in url:
        html_candidate = url
        pdf_candidate = url.replace("/TXT/HTML/", "/TXT/PDF/")
    elif "/TXT/PDF/" in url:
        html_candidate = url.replace("/TXT/PDF/", "/TXT/HTML/")
        pdf_candidate = url
    elif "/TXT/" in url:
        html_candidate = url.replace("/TXT/", "/TXT/HTML/")
        pdf_candidate = url.replace("/TXT/", "/TXT/PDF/")
    elif "/HTML/" in url:
        html_candidate = url.replace("/HTML/", "/TXT/HTML/")
        pdf_candidate = url.replace("/HTML/", "/TXT/PDF/")
    elif "/PDF/" in url:
        html_candidate = url.replace("/PDF/", "/TXT/HTML/")
        pdf_candidate = url.replace("/PDF/", "/TXT/PDF/")

    for candidate in [html_candidate, pdf_candidate]:
        if candidate:
            candidates.append(candidate)

    return list(dict.fromkeys(candidates))


def classify_download_url_candidate(source_url, download_url, order):
    """Human-readable candidate type for auditing fallback behavior."""
    source_url = clean_text(source_url)
    download_url = clean_text(download_url)
    if order == 1 or download_url == source_url:
        return "original"
    if "/TXT/HTML/" in download_url:
        return "eurlex_html_fallback"
    if "/TXT/PDF/" in download_url:
        return "eurlex_pdf_fallback"
    return "fallback"


def infer_extension(url="", content_type=""):
    content_type = clean_text(content_type).lower().split(";")[0]
    if content_type == "application/pdf":
        return "pdf"
    if content_type in {"text/html", "application/xhtml+xml"}:
        return "html"
    if content_type in {"text/plain", "application/xml", "text/xml"}:
        return "txt" if content_type == "text/plain" else "xml"

    parsed_path = unquote(urlparse(clean_text(url)).path)
    suffix = Path(parsed_path).suffix.lower().lstrip(".")
    if suffix in {"pdf", "html", "htm", "txt", "xml", "doc", "docx"}:
        return "html" if suffix == "htm" else suffix

    return "bin"


def make_base_file_stem(row, source_column, source_url):
    identifier = safe_filename(row.get("Identifier", ""), "COM_unknown")
    case_number = safe_filename(row.get("Case number", ""), "case_unknown")
    item_type = safe_filename(row.get("Item type", ""), "item").lower()
    item_label = safe_filename(row.get("Item label", ""), "document").lower()
    item_date = safe_filename(row.get("Item date", ""), "no_date")
    source = {"Item link": "main", "Item OJ link": "secondary_oj", "Item tertiary OJ link": "tertiary_oj"}.get(source_column, "item")
    short_hash = hash_text(source_url, n=10)
    return f"{identifier}_{case_number}_{item_date}_{item_type}_{item_label}_{source}_{short_hash}"


def make_download_key(row, source_column, source_url):
    return hash_text(
        "|".join([
            clean_text(row.get("Identifier", "")),
            clean_text(row.get("Case number", "")),
            clean_text(row.get("Item type", "")),
            clean_text(row.get("Item date", "")),
            clean_text(row.get("Item label", "")),
            source_column,
            source_url,
        ]),
        n=20,
    )


def create_base_download_manifest(view2_df):
    if DOWNLOAD_ONLY_RELEVANT and "Relevant" in view2_df.columns:
        view2_df = view2_df[view2_df["Relevant"].fillna(False).astype(bool)].copy()

    rows = []

    for _, row in view2_df.iterrows():
        item_links = split_manifest_links(row.get("Item link", ""))
        oj_links = split_manifest_links(row.get("Item OJ link", ""))
        tertiary_oj_links = split_manifest_links(row.get("Item tertiary OJ link", ""))

        source_pairs = []
        source_pairs.extend(("Item link", link) for link in item_links)
        source_pairs.extend(("Item OJ link", link) for link in oj_links)

        # Tertiary OJ fallback is only allowed when no row-level URL is present.
        # build_view2 should already enforce this, but the guard below makes the
        # download-manifest logic robust if a CSV is manually edited later.
        if not item_links and not oj_links:
            source_pairs.extend(("Item tertiary OJ link", link) for link in tertiary_oj_links)

        common = {
            "identifier": row.get("Identifier", ""),
            "case_number": row.get("Case number", ""),
            "case_title": row.get("Case title", ""),
            "initiation_of_proceedings": row.get("Initiation of proceedings", ""),
            "item_type": row.get("Item type", ""),
            "item_date": row.get("Item date", ""),
            "decision_key": make_decision_key(row.get("Case number", ""), row.get("Item date", "")),
            "decision_key_reverse": make_decision_key_reverse(row.get("Case number", ""), row.get("Item date", "")),
            "item_label": row.get("Item label", ""),
            "relevant": row.get("Relevant", ""),
        }

        if not source_pairs:
            no_url_key = hash_text(
                "|".join([
                    clean_text(row.get("Identifier", "")),
                    clean_text(row.get("Case number", "")),
                    clean_text(row.get("Item type", "")),
                    clean_text(row.get("Item date", "")),
                    clean_text(row.get("Item label", "")),
                    "no_download_available",
                ]),
                n=20,
            )
            rows.append({
                "download_key": no_url_key,
                **common,
                "source_column": "",
                "source_url": "",
                "source_role": "none",
                "download_url": "",
                "download_url_kind": "",
                "source_url_candidates_json": "[]",
                "url_candidate_order": pd.NA,
                "download_file_name": "",
                "download_path": pd.NA,
                "download_success": False,
                "download_status": "no_download_available",
                "http_status": pd.NA,
                "content_type": pd.NA,
                "bytes": pd.NA,
                "error": "Neither Item link, Item OJ link, nor tertiary case-level OJ link is available for this View 2 row.",
            })
            continue

        for source_column, source_url in source_pairs:
            candidate_urls = expand_download_url_candidates(source_url)
            candidate_urls_json = json.dumps(candidate_urls, ensure_ascii=False)
            for order, download_url in enumerate(candidate_urls, start=1):
                candidate_kind = classify_download_url_candidate(source_url, download_url, order)
                stem = make_base_file_stem(row, source_column, source_url)

                # Important: each EUR-Lex/OJ fallback candidate must have its own
                # local path. Otherwise a failed/successful original /TXT/ row can
                # mask the /TXT/HTML/ and /TXT/PDF/ fallback rows during the
                # local-file sync step.
                if len(candidate_urls) > 1:
                    stem = f"{stem}_candidate{order}_{safe_filename(candidate_kind, 'candidate')}"

                ext = infer_extension(download_url)
                file_name = safe_filename(f"{stem}.{ext}", default=f"{stem}.{ext}")

                rows.append({
                    "download_key": make_download_key(row, source_column, source_url),
                    **common,
                    "source_column": source_column,
                    "source_url": source_url,
                    "source_role": {"Item link": "main", "Item OJ link": "secondary_oj", "Item tertiary OJ link": "tertiary_oj"}.get(source_column, "other"),
                    "download_url": download_url,
                    "download_url_kind": candidate_kind,
                    "source_url_candidates_json": candidate_urls_json,
                    "url_candidate_order": order,
                    "download_file_name": file_name,
                    "download_path": str(COM_ANTITRUST_DATA_DIR / file_name),
                    "download_success": pd.NA,
                    "download_status": "pending",
                    "http_status": pd.NA,
                    "content_type": pd.NA,
                    "bytes": pd.NA,
                    "error": pd.NA,
                })

    df = pd.DataFrame(rows)
    for col in DOWNLOAD_MANIFEST_COLUMNS:
        if col not in df.columns:
            df[col] = pd.NA
    df = df[DOWNLOAD_MANIFEST_COLUMNS]

    # Deduplicate exact candidate URLs. Keep no-download rows separate.
    downloadable = df[df["download_status"] != "no_download_available"].copy()
    no_download = df[df["download_status"] == "no_download_available"].copy()
    downloadable = downloadable.drop_duplicates(
        subset=["download_key", "download_url", "url_candidate_order"],
        keep="first",
    )
    df = pd.concat([downloadable, no_download], ignore_index=True)
    return df


def save_download_manifest(df):
    df.to_csv(DOWNLOAD_MANIFEST_CSV, index=False, encoding="utf-8-sig")

    parquet_df = df.copy()
    for col in ["http_status", "bytes", "url_candidate_order"]:
        if col in parquet_df.columns:
            parquet_df[col] = pd.to_numeric(parquet_df[col].replace("", pd.NA), errors="coerce").astype("Int64")
    if "download_success" in parquet_df.columns:
        parquet_df["download_success"] = parquet_df["download_success"].astype("boolean")
    for col in parquet_df.columns:
        if parquet_df[col].dtype == object:
            parquet_df[col] = parquet_df[col].astype("string")

    try:
        parquet_df.to_parquet(DOWNLOAD_MANIFEST_PARQUET, index=False)
    except Exception as e:
        logger.warning("Could not save parquet download manifest %s: %s", DOWNLOAD_MANIFEST_PARQUET, e)


def normalize_download_path(file_name):
    """Always point manifest rows to data/raw/com_antitrust, even if an older manifest used data/com_antitrust."""
    file_name = clean_text(file_name)
    if not file_name:
        return pd.NA
    return str(COM_ANTITRUST_DATA_DIR / Path(file_name).name)


def find_existing_local_file(path_value):
    """
    Check the intended local path on disk.

    If the manifest currently points to a pre-download placeholder such as .bin,
    also check common final extensions with the same stem. This matters because
    the downloader may correct the suffix after seeing the response Content-Type
    (.html, .pdf, .txt, .xml).
    """
    path_text = clean_text(path_value)
    if not path_text:
        return None, 0

    path = Path(path_text)
    candidates = [path]

    # If the exact path is a placeholder or has no useful suffix, search likely
    # final suffixes generated by download_one_manifest_row().
    suffixes = [".html", ".pdf", ".txt", ".xml", ".bin"]
    for suffix in suffixes:
        candidates.append(path.with_suffix(suffix))

    # Preserve order while deduplicating paths.
    seen = set()
    ordered = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            ordered.append(candidate)
            seen.add(key)

    for candidate in ordered:
        try:
            if candidate.exists() and candidate.is_file() and candidate.stat().st_size > 0:
                return candidate, candidate.stat().st_size
        except OSError:
            continue

    return None, 0


def sync_manifest_with_local_files(df, reset_missing_success=True):
    """
    Reconcile the manifest with the physical files on disk.

    This deliberately does not trust the old manifest alone. Before downloading,
    every row checks whether its intended local file is already present in
    data/raw/com_antitrust. If the file exists, the row is marked as already_exists
    and will be skipped. If an old manifest says downloaded/already_exists but the
    file is missing, the row is reset to pending so it can be retried.
    """
    df = df.copy()

    for col in ["local_file_exists", "local_file_bytes"]:
        if col not in df.columns:
            df[col] = pd.NA

    for idx, row in df.iterrows():
        status = clean_text(row.get("download_status", ""))
        url = clean_text(row.get("download_url", ""))
        path_text = clean_text(row.get("download_path", ""))

        if status == "no_download_available" or not path_text:
            df.at[idx, "local_file_exists"] = False
            df.at[idx, "local_file_bytes"] = 0
            continue

        existing_path, existing_size = find_existing_local_file(path_text)
        file_exists = existing_path is not None and existing_size > 0

        df.at[idx, "local_file_exists"] = bool(file_exists)
        df.at[idx, "local_file_bytes"] = int(existing_size) if file_exists else 0

        if file_exists:
            # Update to the actual file found on disk. This handles .bin -> .html/.pdf.
            df.at[idx, "download_file_name"] = existing_path.name
            df.at[idx, "download_path"] = str(existing_path)
            df.at[idx, "download_success"] = True
            df.at[idx, "download_status"] = "already_exists"
            df.at[idx, "bytes"] = int(existing_size)
            df.at[idx, "error"] = pd.NA
            continue

        if reset_missing_success and status in {"downloaded", "already_exists"} and url.startswith("http"):
            df.at[idx, "download_success"] = pd.NA
            df.at[idx, "download_status"] = "pending"
            df.at[idx, "http_status"] = pd.NA
            df.at[idx, "content_type"] = pd.NA
            df.at[idx, "bytes"] = pd.NA
            df.at[idx, "error"] = "Previous manifest said file existed, but no local file was found. Reset for retry."

    return df


def merge_existing_download_progress(base_df):
    """
    Preserve previous download statuses by download_key + download_url, but do NOT preserve
    stale download_path values from older notebooks. All paths are rebuilt from
    COM_ANTITRUST_DATA_DIR, i.e. data/raw/com_antitrust.
    """
    if not DOWNLOAD_MANIFEST_CSV.exists():
        base_df["download_path"] = base_df["download_file_name"].apply(normalize_download_path)
        base_df.loc[base_df["download_status"] == "no_download_available", "download_path"] = pd.NA
        return sync_manifest_with_local_files(base_df)

    existing = pd.read_csv(DOWNLOAD_MANIFEST_CSV, low_memory=False)
    key_cols = ["download_key", "download_url"]

    # Keep the existing result metadata, but intentionally exclude download_path so old
    # data/com_antitrust paths cannot leak into the refreshed manifest.
    result_cols = ["download_file_name", "download_success", "download_status", "http_status", "content_type", "bytes", "error"]

    for col in key_cols + result_cols:
        if col not in existing.columns:
            existing[col] = pd.NA

    existing_progress = existing[key_cols + result_cols].drop_duplicates(subset=key_cols, keep="last")
    merged = base_df.merge(existing_progress, on=key_cols, how="left", suffixes=("", "_existing"))

    for col in result_cols:
        ex = f"{col}_existing"
        if ex not in merged.columns:
            continue
        # Preserve existing non-empty values, but never overwrite current no-download rows.
        mask = (
            merged[ex].notna()
            & (merged[ex].astype(str) != "")
            & (merged["download_status"] != "no_download_available")
        )
        merged.loc[mask, col] = merged.loc[mask, ex]
        merged = merged.drop(columns=[ex])

    # Rebuild paths after merging, using the current target directory only.
    merged["download_path"] = merged["download_file_name"].apply(normalize_download_path)
    merged.loc[merged["download_status"] == "no_download_available", "download_path"] = pd.NA

    # Crucial: do not rely on old manifest statuses only. Re-check the actual
    # files in data/raw/com_antitrust before deciding what to download.
    merged = sync_manifest_with_local_files(merged)

    return merged[DOWNLOAD_MANIFEST_COLUMNS]


# Build a fresh base manifest from View 2.
# v14: the CSV is no longer treated as authoritative progress. Progress is
# reconstructed later from download_attempts.jsonl plus a filesystem check.
base_download_manifest = create_base_download_manifest(view2)
base_download_manifest["download_path"] = base_download_manifest["download_file_name"].apply(normalize_download_path)
base_download_manifest.loc[base_download_manifest["download_status"] == "no_download_available", "download_path"] = pd.NA

download_manifest = sync_manifest_with_local_files(base_download_manifest, reset_missing_success=True)

print("Download manifest target:", DOWNLOAD_MANIFEST_CSV)
print("Download JSONL log:", DOWNLOAD_ATTEMPTS_JSONL)
print("Mode:", DOWNLOAD_MODE)
print("Rows:", len(download_manifest))
print(download_manifest["download_status"].fillna("MISSING").value_counts(dropna=False))

print("\nManifest rows by source role:")
if "source_role" in download_manifest.columns:
    display(download_manifest["source_role"].fillna("MISSING").value_counts(dropna=False))

print("\nEUR-Lex/OJ candidate expansion by source role and candidate kind:")
if {"source_role", "download_url_kind"}.issubset(download_manifest.columns):
    display(pd.crosstab(download_manifest["source_role"].fillna("MISSING"), download_manifest["download_url_kind"].fillna("MISSING")))

print("\nSample tertiary OJ fallback rows, if any:")
if {"source_role", "source_url", "download_url", "download_url_kind", "url_candidate_order"}.issubset(download_manifest.columns):
    display(download_manifest.loc[
        download_manifest["source_role"].eq("tertiary_oj"),
        ["case_number", "item_date", "item_label", "source_url", "url_candidate_order", "download_url_kind", "download_url", "download_status"]
    ].head(30))

print("\nPending downloadable rows before JSONL replay/download cell:")
pending_downloadable = download_manifest[
    download_manifest["download_status"].fillna("").astype(str).eq("pending")
    & download_manifest["download_url"].fillna("").astype(str).str.startswith("http")
].copy()
if len(pending_downloadable):
    display(pending_downloadable["source_role"].fillna("MISSING").value_counts(dropna=False))
else:
    print("No pending downloadable rows.")

print("Local file existence check:")
if "local_file_exists" in download_manifest.columns:
    print(download_manifest["local_file_exists"].fillna(False).value_counts(dropna=False))

missing_success = download_manifest[
    download_manifest["download_status"].isin(["downloaded", "already_exists"])
    & (~download_manifest["local_file_exists"].fillna(False).astype(bool))
]
print("Rows marked successful but missing on disk after sync:", len(missing_success))

case_download_availability = (
    download_manifest
    .assign(has_download_url=lambda d: d["download_url"].fillna("").astype(str).str.startswith("http"))
    .groupby(["case_number", "case_title"], dropna=False)
    .agg(
        manifest_rows=("download_key", "count"),
        downloadable_rows=("has_download_url", "sum"),
    )
    .reset_index()
)
cases_with_nothing_to_download = case_download_availability[case_download_availability["downloadable_rows"] == 0].copy()

print("Cases with nothing to download:", len(cases_with_nothing_to_download))
display(cases_with_nothing_to_download.head(50))


Download manifest target: /home/edik/projects/eccjeu/output/com_antitrust/download_manifest.csv
Download JSONL log: /home/edik/projects/eccjeu/output/com_antitrust/download_attempts.jsonl
Mode: update_jsonl
Rows: 2492
download_status
pending                  1303
already_exists           1174
no_download_available      15
Name: count, dtype: int64

Manifest rows by source role:


source_role
secondary_oj    1755
main             593
tertiary_oj      129
none              15
Name: count, dtype: int64


EUR-Lex/OJ candidate expansion by source role and candidate kind:


download_url_kind,,eurlex_html_fallback,eurlex_pdf_fallback,original
source_role,,,,
main,0,0,0,593
none,15,0,0,0
secondary_oj,0,585,585,585
tertiary_oj,0,43,43,43



Sample tertiary OJ fallback rows, if any:


...


Pending downloadable rows before JSONL replay/download cell:


source_role
secondary_oj    1217
tertiary_oj       86
Name: count, dtype: int64

Local file existence check:
local_file_exists
False    1318
True     1174
Name: count, dtype: int64
Rows marked successful but missing on disk after sync: 0
Cases with nothing to download: 6


/tmp/ipykernel_506059/1362238252.py:589: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print(download_manifest["local_file_exists"].fillna(False).value_counts(dropna=False))
/tmp/ipykernel_506059/1362238252.py:593: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  & (~download_manifest["local_file_exists"].fillna(False).astype(bool))


,case_number,case_title,manifest_rows,downloadable_rows
57,AT.36246,Marathon/Ruhrgas/GDF et alia,1,0
66,AT.36533,Yves Saint Laurent,1,0
69,AT.36559,British Gas+Wingas+1,1,0
80,AT.36817,PO/Nederlandse Vissersbond,1,0
284,AT.40054,Ethanol benchmarks,2,0
319,AT.40545,Automotive Starter Batteries,1,0


### EUR-Lex URL expansion sanity check

This quick check confirms that plain `/EN/TXT/` EUR-Lex links are converted only to nested `/EN/TXT/HTML/` and `/EN/TXT/PDF/` candidates.


In [10]:
test_eurlex_url = "https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.L_.1999.125.01.0012.01.ENG&toc=OJ:L:1999:125:TOC"
for i, candidate in enumerate(expand_download_url_candidates(test_eurlex_url), start=1):
    print(i, candidate)

assert expand_download_url_candidates(test_eurlex_url) == [
    test_eurlex_url,
    test_eurlex_url.replace("/TXT/", "/TXT/HTML/"),
    test_eurlex_url.replace("/TXT/", "/TXT/PDF/"),
]
print("EUR-Lex expansion OK: original URL first, then /TXT/HTML/ and /TXT/PDF/ fallbacks.")


1 https://eur-lex.europa.eu/legal-content/EN/TXT/?uri=uriserv:OJ.L_.1999.125.01.0012.01.ENG&toc=OJ:L:1999:125:TOC
2 https://eur-lex.europa.eu/legal-content/EN/TXT/HTML/?uri=uriserv:OJ.L_.1999.125.01.0012.01.ENG&toc=OJ:L:1999:125:TOC
3 https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=uriserv:OJ.L_.1999.125.01.0012.01.ENG&toc=OJ:L:1999:125:TOC
EUR-Lex expansion OK: original URL first, then /TXT/HTML/ and /TXT/PDF/ fallbacks.


## 7. JSONL-backed download status and downloads

V14 makes `download_attempts.jsonl` the authoritative progress log and keeps the existing CSV/parquet outputs as regenerated reports. The three modes are:

- `rebuild_jsonl`: overwrite JSONL, check local files, then redownload anything still missing.
- `update_jsonl`: keep JSONL, check local files, then only try rows not already successful.
- `update_csv_from_jsonl`: no downloads; overwrite the CSV/parquet from JSONL plus the current filesystem check.

The output filenames stay the same as before, except for the added JSONL log.


## v14 retry profile note

EUR-Lex `202 Accepted` handling is deliberately slower than v13: more retries, longer backoff, and a higher maximum wait before moving to the next candidate URL.


In [11]:
import random

def mark_candidate_group_skipped_after_success(df, success_idx):
    """After a source URL succeeds, skip remaining candidates for that source URL."""
    success_row = df.loc[success_idx]
    group_mask = (
        (df["download_key"] == success_row["download_key"])
        & (df["source_url"] == success_row["source_url"])
        & (df.index != success_idx)
        & (df["download_status"].fillna("") == "pending")
    )
    df.loc[group_mask, "download_success"] = False
    df.loc[group_mask, "download_status"] = "skipped_candidate_after_success"
    df.loc[group_mask, "error"] = "Another candidate URL for this source URL downloaded successfully."
    return df


def should_attempt_row(row):
    # First check the actual file system, not only the manifest status.
    # If the local file already exists, this row should not be downloaded again,
    # even if an old manifest status says failed/pending.
    output_path = clean_text(row.get("download_path", ""))
    existing_path, existing_size = find_existing_local_file(output_path)
    if existing_path is not None and existing_size > 0:
        return False

    status = clean_text(row.get("download_status", ""))
    if status in {"downloaded", "already_exists", "no_download_available", "skipped_candidate_after_success"}:
        return False
    url = clean_text(row.get("download_url", ""))
    return url.startswith("http")


def _eurlex_headers_for_url(url):
    """Use browser-like Accept headers for EUR-Lex HTML/PDF fallbacks."""
    headers = dict(HEADERS)
    if "/TXT/PDF/" in clean_text(url):
        headers["Accept"] = "application/pdf,application/octet-stream;q=0.9,*/*;q=0.8"
    elif "/TXT/HTML/" in clean_text(url):
        headers["Accept"] = "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8"
    else:
        headers["Accept"] = "text/html,application/xhtml+xml,application/xml;q=0.9,application/pdf;q=0.8,*/*;q=0.7"
    return headers


def _looks_like_usable_document(content, content_type="", url=""):
    """Reject empty/placeholder EUR-Lex responses that are not actual documents."""
    if not content:
        return False
    ctype = clean_text(content_type).lower()
    if len(content) < 200:
        # A few-byte 202/placeholder response is not useful evidence.
        return False
    if "/TXT/PDF/" in clean_text(url) and b"%PDF" not in content[:1024] and "pdf" not in ctype:
        return False
    return True


def _request_with_eurlex_202_handling(session, url):
    """
    Fetch one candidate URL with special handling for EUR-Lex HTTP 202.

    EUR-Lex can return 202 Accepted while it prepares dynamically generated
    HTML/PDF/TXT representations. For EUR-Lex legal-content URLs this function:
    1. uses browser-like request headers;
    2. waits briefly before the request to avoid hammering EUR-Lex;
    3. if 202 is returned, honours Retry-After where present;
    4. otherwise retries with exponential backoff;
    5. follows Location / Content-Location if provided;
    6. returns the final response to the caller.

    Downloads remain sequential because run_downloads iterates rows one by one
    with a single requests.Session.
    """
    url = clean_text(url)
    is_eurlex = "eur-lex.europa.eu/legal-content/" in url
    headers = _eurlex_headers_for_url(url)

    if is_eurlex and EURLEX_PRE_REQUEST_SLEEP_SECONDS:
        time.sleep(EURLEX_PRE_REQUEST_SLEEP_SECONDS)

    response = session.get(url, headers=headers, timeout=DOWNLOAD_TIMEOUT_SECONDS, allow_redirects=True)

    if not is_eurlex:
        return response, 0, ""

    retries_used = 0
    seen_urls = {url}
    notes = []

    while response.status_code == 202 and retries_used < EURLEX_202_MAX_RETRIES:
        retries_used += 1

        retry_after = response.headers.get("Retry-After")
        if retry_after:
            try:
                sleep_seconds = min(float(retry_after), EURLEX_202_MAX_SLEEP_SECONDS)
                notes.append(f"Retry-After={retry_after}")
            except Exception:
                sleep_seconds = min(
                    EURLEX_202_SLEEP_SECONDS * (EURLEX_202_BACKOFF_FACTOR ** (retries_used - 1)),
                    EURLEX_202_MAX_SLEEP_SECONDS,
                )
        else:
            sleep_seconds = min(
                EURLEX_202_SLEEP_SECONDS * (EURLEX_202_BACKOFF_FACTOR ** (retries_used - 1)),
                EURLEX_202_MAX_SLEEP_SECONDS,
            )

        time.sleep(sleep_seconds)

        next_url = response.headers.get("Location") or response.headers.get("Content-Location") or url
        if next_url.startswith("/"):
            parsed = urlparse(url)
            next_url = f"{parsed.scheme}://{parsed.netloc}{next_url}"

        # Avoid infinite Location loops. If Location repeats, retry the original candidate.
        if next_url in seen_urls and next_url != url:
            next_url = url
        seen_urls.add(next_url)

        response = session.get(next_url, headers=headers, timeout=DOWNLOAD_TIMEOUT_SECONDS, allow_redirects=True)

    if retries_used:
        notes.insert(0, f"EUR-Lex 202 retries used: {retries_used}")
    note = "; ".join(notes)
    return response, retries_used, note


def download_one_manifest_row(session, row, overwrite=False):
    url = clean_text(row.get("download_url", ""))
    output_path = Path(clean_text(row.get("download_path", "")))

    if not url:
        return {
            "download_success": False,
            "download_status": "failed_no_url",
            "http_status": pd.NA,
            "content_type": pd.NA,
            "bytes": pd.NA,
            "error": "No download_url available.",
        }

    if output_path.exists() and not overwrite:
        return {
            "download_success": True,
            "download_status": "already_exists",
            "http_status": pd.NA,
            "content_type": pd.NA,
            "bytes": output_path.stat().st_size,
            "error": pd.NA,
        }

    try:
        response, retries_used, retry_note = _request_with_eurlex_202_handling(session, url)
        status = response.status_code
        content_type = response.headers.get("Content-Type", "")
        content = response.content or b""

        if status != 200:
            # 202 Accepted is not a usable document. The outer loop will continue
            # to the next candidate URL for this same source_url.
            status_label = "failed_http_202_accepted" if status == 202 else "failed_http_status"
            error_msg = f"HTTP status {status}"
            if retry_note:
                error_msg = f"{error_msg}; {retry_note}"
            return {
                "download_success": False,
                "download_status": status_label,
                "http_status": status,
                "content_type": content_type,
                "bytes": len(content),
                "error": error_msg,
            }

        if not _looks_like_usable_document(content, content_type=content_type, url=url):
            return {
                "download_success": False,
                "download_status": "failed_empty_or_placeholder_response",
                "http_status": status,
                "content_type": content_type,
                "bytes": len(content),
                "error": f"Downloaded response was empty or looked like a placeholder. {retry_note}".strip(),
            }

        # Correct extension after seeing the Content-Type.
        ext = infer_extension(url=url, content_type=content_type)
        if ext and output_path.suffix.lower().lstrip(".") != ext:
            output_path = output_path.with_suffix("." + ext)

        output_path.parent.mkdir(parents=True, exist_ok=True)
        with open(output_path, "wb") as f:
            f.write(content)

        return {
            "download_success": True,
            "download_status": "downloaded",
            "http_status": status,
            "content_type": content_type,
            "bytes": len(content),
            "error": retry_note or pd.NA,
            "download_path_actual": str(output_path),
            "download_file_name_actual": output_path.name,
        }

    except Exception as e:
        return {
            "download_success": False,
            "download_status": "failed_exception",
            "http_status": pd.NA,
            "content_type": pd.NA,
            "bytes": pd.NA,
            "error": repr(e),
        }

def run_downloads(download_manifest, limit=None, overwrite=False, save_every=25):
    # Re-check actual files on disk immediately before downloading. This prevents
    # unnecessary retries when the manifest is stale but the file exists locally.
    df = sync_manifest_with_local_files(download_manifest.copy())
    save_download_manifest(df)

    pending_indices = [idx for idx, row in df.iterrows() if should_attempt_row(row)]
    if limit is not None:
        pending_indices = pending_indices[:limit]

    print(f"Rows to attempt: {len(pending_indices):,}")

    session = requests.Session()
    session.headers.update(HEADERS)
    print("Download mode: sequential, one candidate URL at a time, using one browser-like requests.Session.")

    for n, idx in enumerate(tqdm(pending_indices, desc="Downloading COM antitrust documents"), start=1):
        # This candidate may have been skipped because an earlier candidate in the same group succeeded.
        if not should_attempt_row(df.loc[idx]):
            continue

        result = download_one_manifest_row(session, df.loc[idx], overwrite=overwrite)

        for key, value in result.items():
            if key in df.columns:
                df.at[idx, key] = value

        # download_one_manifest_row may correct the extension after seeing
        # Content-Type. Persist the actual final path/name back to the manifest.
        if result.get("download_path_actual"):
            df.at[idx, "download_path"] = result.get("download_path_actual")
        if result.get("download_file_name_actual"):
            df.at[idx, "download_file_name"] = result.get("download_file_name_actual")

        if result.get("download_success") is True:
            df = mark_candidate_group_skipped_after_success(df, idx)

        if save_every and n % save_every == 0:
            save_download_manifest(df)

        # v12: moderate row-level pause. Keep normal throughput reasonable; only
        # slow down heavily when the server explicitly blocks or throttles.
        status = result.get("http_status")
        try:
            status_int = int(status) if pd.notna(status) else None
        except Exception:
            status_int = None

        if status_int == 403:
            time.sleep(random.uniform(*HTTP_403_SLEEP_SECONDS))
        elif status_int == 429:
            time.sleep(random.uniform(*HTTP_429_SLEEP_SECONDS))
        else:
            time.sleep(random.uniform(*DOWNLOAD_SLEEP_SECONDS))

    save_download_manifest(df)
    return df





def _download_success_bool_series(s):
    if isinstance(s, pd.Series):
        if pd.api.types.is_bool_dtype(s):
            return s.fillna(False)
        return s.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
    return str(s).strip().lower() in {"true", "1", "yes", "y"}


def _truthy(value):
    if value is pd.NA or value is None:
        return False
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}

def _json_safe(value):
    """Convert pandas/numpy/path values into JSONL-safe primitives."""
    if value is pd.NA or value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if isinstance(value, Path):
        return str(value)
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            pass
    return value


def row_identity_payload(row):
    """Stable manifest identity fields copied into each JSONL event for auditing."""
    fields = [
        "download_key", "identifier", "case_number", "case_title", "item_type", "item_date",
        "decision_key", "decision_key_reverse", "item_label", "relevant", "source_column",
        "source_url", "source_role", "download_url", "download_url_kind", "url_candidate_order",
        "download_file_name", "download_path",
    ]
    return {field: _json_safe(row.get(field, None)) for field in fields}


def make_download_event(row, result, event_type="download_attempt"):
    event = {
        "event_ts": datetime.now().isoformat(timespec="seconds"),
        "event_type": event_type,
        **row_identity_payload(row),
    }
    for key in ["download_success", "download_status", "http_status", "content_type", "bytes", "error", "local_file_exists", "local_file_bytes"]:
        if key in result:
            event[key] = _json_safe(result.get(key))
    if "download_path_actual" in result:
        event["download_path"] = _json_safe(result.get("download_path_actual"))
    if "download_file_name_actual" in result:
        event["download_file_name"] = _json_safe(result.get("download_file_name_actual"))
    return event


def append_jsonl_event(event):
    DOWNLOAD_ATTEMPTS_JSONL.parent.mkdir(parents=True, exist_ok=True)
    with open(DOWNLOAD_ATTEMPTS_JSONL, "a", encoding="utf-8") as f:
        f.write(json.dumps(event, ensure_ascii=False, default=str) + "\n")


def load_download_events():
    if not DOWNLOAD_ATTEMPTS_JSONL.exists():
        return pd.DataFrame()
    rows = []
    with open(DOWNLOAD_ATTEMPTS_JSONL, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
                row["jsonl_line_no"] = line_no
                rows.append(row)
            except Exception as e:
                logger.warning("Skipping malformed JSONL line %s in %s: %s", line_no, DOWNLOAD_ATTEMPTS_JSONL, e)
    return pd.DataFrame(rows)


def latest_jsonl_status_by_candidate(events_df):
    if events_df.empty:
        return pd.DataFrame()
    key_cols = ["download_key", "download_url"]
    for col in key_cols:
        if col not in events_df.columns:
            events_df[col] = pd.NA
    sort_cols = [c for c in ["event_ts", "jsonl_line_no"] if c in events_df.columns]
    latest = events_df.sort_values(sort_cols).drop_duplicates(subset=key_cols, keep="last") if sort_cols else events_df.drop_duplicates(subset=key_cols, keep="last")
    keep_cols = key_cols + [c for c in [
        "download_file_name", "download_path", "download_success", "download_status", "http_status",
        "content_type", "bytes", "error", "local_file_exists", "local_file_bytes", "event_ts", "event_type",
    ] if c in latest.columns]
    return latest[keep_cols].copy()


def apply_jsonl_status_to_manifest(base_df, events_df):
    """Rebuild the manifest status from JSONL, then verify against local files."""
    df = base_df.copy()
    latest = latest_jsonl_status_by_candidate(events_df)
    if not latest.empty:
        key_cols = ["download_key", "download_url"]
        progress_cols = [c for c in latest.columns if c not in key_cols]
        rename = {c: f"{c}__jsonl" for c in progress_cols}
        merged = df.merge(latest.rename(columns=rename), on=key_cols, how="left")
        for col in progress_cols:
            src_col = f"{col}__jsonl"
            if src_col not in merged.columns:
                continue
            if col in ["download_path", "download_file_name"]:
                mask = merged[src_col].notna() & merged[src_col].astype(str).str.strip().ne("")
            else:
                mask = merged[src_col].notna()
            if col in merged.columns:
                merged.loc[mask, col] = merged.loc[mask, src_col]
            else:
                merged[col] = merged[src_col]
            merged = merged.drop(columns=[src_col])
        df = merged

    # Filesystem is always checked last because it is the real state on disk.
    df = sync_manifest_with_local_files(df, reset_missing_success=True)
    return df[DOWNLOAD_MANIFEST_COLUMNS]


def append_filesystem_events_for_found_files(df, existing_events_df=None):
    """Add JSONL entries when v14 discovers files on disk that JSONL did not yet know about."""
    if existing_events_df is None:
        existing_events_df = load_download_events()
    latest = latest_jsonl_status_by_candidate(existing_events_df)
    success_keys = set()
    if not latest.empty and {"download_key", "download_url", "download_status"}.issubset(latest.columns):
        tmp = latest.copy()
        tmp["success_bool"] = _download_success_bool_series(tmp.get("download_success", False)) | tmp["download_status"].fillna("").isin({"downloaded", "already_exists"})
        success_keys = set(map(tuple, tmp.loc[tmp["success_bool"], ["download_key", "download_url"]].astype(str).values.tolist()))

    appended = 0
    for _, row in df.iterrows():
        key = (str(row.get("download_key", "")), str(row.get("download_url", "")))
        if key in success_keys:
            continue
        if _truthy(row.get("local_file_exists", False)) and clean_text(row.get("download_url", "")).startswith("http"):
            result = {
                "download_success": True,
                "download_status": "already_exists",
                "http_status": pd.NA,
                "content_type": pd.NA,
                "bytes": row.get("local_file_bytes", row.get("bytes", pd.NA)),
                "error": pd.NA,
                "local_file_exists": True,
                "local_file_bytes": row.get("local_file_bytes", pd.NA),
            }
            append_jsonl_event(make_download_event(row, result, event_type="filesystem_found"))
            appended += 1
    return appended


def rebuild_csv_from_jsonl(base_df):
    events = load_download_events()
    df = apply_jsonl_status_to_manifest(base_df, events)
    appended = append_filesystem_events_for_found_files(df, existing_events_df=events)
    if appended:
        events = load_download_events()
        df = apply_jsonl_status_to_manifest(base_df, events)
    save_download_manifest(df)
    return df


def run_downloads_jsonl(base_df, mode="update_jsonl", limit=None, overwrite=False, save_every=25):
    mode = clean_text(mode).lower()
    valid_modes = {"rebuild_jsonl", "update_jsonl", "update_csv_from_jsonl"}
    if mode not in valid_modes:
        raise ValueError(f"DOWNLOAD_MODE must be one of {sorted(valid_modes)}, got: {mode!r}")

    if mode == "rebuild_jsonl":
        if DOWNLOAD_ATTEMPTS_JSONL.exists():
            DOWNLOAD_ATTEMPTS_JSONL.unlink()
        print("Rebuilding JSONL from scratch:", DOWNLOAD_ATTEMPTS_JSONL)
    elif mode == "update_jsonl":
        print("Updating JSONL:", DOWNLOAD_ATTEMPTS_JSONL)
    else:
        print("Rebuilding CSV/parquet from JSONL without downloading:", DOWNLOAD_ATTEMPTS_JSONL)

    events = load_download_events()
    df = apply_jsonl_status_to_manifest(base_df, events)
    found_count = append_filesystem_events_for_found_files(df, existing_events_df=events)
    if found_count:
        print(f"Filesystem check added {found_count:,} JSONL already_exists events.")
        events = load_download_events()
        df = apply_jsonl_status_to_manifest(base_df, events)

    if mode == "update_csv_from_jsonl" or not DOWNLOAD_FILES:
        save_download_manifest(df)
        if not DOWNLOAD_FILES and mode != "update_csv_from_jsonl":
            print("DOWNLOAD_FILES is False. JSONL/CSV were synced, but no network downloads were attempted.")
        return df

    pending_indices = [idx for idx, row in df.iterrows() if should_attempt_row(row)]
    if limit is not None:
        pending_indices = pending_indices[:limit]

    print(f"Rows to attempt: {len(pending_indices):,}")
    session = requests.Session()
    session.headers.update(HEADERS)
    print("Download mode: sequential, one candidate URL at a time, using one browser-like requests.Session.")

    for n, idx in enumerate(tqdm(pending_indices, desc="Downloading COM antitrust documents"), start=1):
        if not should_attempt_row(df.loc[idx]):
            continue

        result = download_one_manifest_row(session, df.loc[idx], overwrite=overwrite)
        event = make_download_event(df.loc[idx], result, event_type="download_attempt")
        append_jsonl_event(event)

        for key, value in result.items():
            if key in df.columns:
                df.at[idx, key] = value
        if result.get("download_path_actual"):
            df.at[idx, "download_path"] = result.get("download_path_actual")
        if result.get("download_file_name_actual"):
            df.at[idx, "download_file_name"] = result.get("download_file_name_actual")

        # Re-check the local file immediately and log it if found.
        row_after = sync_manifest_with_local_files(df.loc[[idx]], reset_missing_success=False).iloc[0]
        for col in ["download_file_name", "download_path", "download_success", "download_status", "bytes", "local_file_exists", "local_file_bytes", "error"]:
            if col in df.columns:
                df.at[idx, col] = row_after.get(col, df.at[idx, col])

        if result.get("download_success") is True:
            df = mark_candidate_group_skipped_after_success(df, idx)
            # Log the skipped candidate statuses too, so CSV rebuilds reproduce the same story.
            skipped_mask = (
                (df["download_key"] == df.at[idx, "download_key"])
                & (df["source_url"] == df.at[idx, "source_url"])
                & (df.index != idx)
                & (df["download_status"].fillna("") == "skipped_candidate_after_success")
            )
            for skipped_idx in df.index[skipped_mask]:
                skipped_result = {
                    "download_success": False,
                    "download_status": "skipped_candidate_after_success",
                    "http_status": pd.NA,
                    "content_type": pd.NA,
                    "bytes": pd.NA,
                    "error": "Another candidate URL for this source URL downloaded successfully.",
                    "local_file_exists": False,
                    "local_file_bytes": 0,
                }
                append_jsonl_event(make_download_event(df.loc[skipped_idx], skipped_result, event_type="candidate_skipped_after_success"))

        if save_every and n % save_every == 0:
            df = rebuild_csv_from_jsonl(base_df)

        status = result.get("http_status")
        try:
            status_int = int(status) if pd.notna(status) else None
        except Exception:
            status_int = None

        if status_int == 403:
            time.sleep(random.uniform(*HTTP_403_SLEEP_SECONDS))
        elif status_int == 429:
            time.sleep(random.uniform(*HTTP_429_SLEEP_SECONDS))
        elif status_int == 202:
            # Extra courtesy pause after unresolved 202 responses.
            time.sleep(random.uniform(max(DOWNLOAD_SLEEP_SECONDS[0], 5.0), max(DOWNLOAD_SLEEP_SECONDS[1], 10.0)))
        else:
            time.sleep(random.uniform(*DOWNLOAD_SLEEP_SECONDS))

    df = rebuild_csv_from_jsonl(base_df)
    return df


# Execute selected v14 mode.
download_manifest = run_downloads_jsonl(
    base_download_manifest,
    mode=DOWNLOAD_MODE,
    limit=DOWNLOAD_LIMIT,
    overwrite=OVERWRITE_EXISTING_FILES,
    save_every=SAVE_EVERY,
)

print("Download status counts:")
display(download_manifest["download_status"].fillna("MISSING").value_counts(dropna=False))

print("\nDownload status by source role:")
if "source_role" in download_manifest.columns:
    display(pd.crosstab(
        download_manifest["source_role"].fillna("MISSING"),
        download_manifest["download_status"].fillna("MISSING"),
        dropna=False,
    ))

remaining_pending = download_manifest[
    download_manifest["download_status"].fillna("").astype(str).eq("pending")
    & download_manifest["download_url"].fillna("").astype(str).str.startswith("http")
]
print("\nRemaining pending downloadable rows after v14 mode:", len(remaining_pending))
if len(remaining_pending):
    display(remaining_pending[["case_number", "item_date", "item_label", "source_role", "download_url", "download_status"]].head(50))

failed_downloads = download_manifest[
    download_manifest["download_status"].fillna("").astype(str).str.startswith("failed")
].copy()

print("Failed rows:", len(failed_downloads))
if len(failed_downloads):
    display(failed_downloads[[
        "identifier",
        "case_number",
        "case_title",
        "item_type",
        "item_date",
        "item_label",
        "source_column",
        "download_url",
        "download_status",
        "http_status",
        "error",
    ]].head(100))

cases_with_failed_downloads = (
    failed_downloads[["case_number", "case_title"]]
    .drop_duplicates()
    .sort_values(["case_number", "case_title"], na_position="last")
) if len(failed_downloads) else pd.DataFrame(columns=["case_number", "case_title"])
print("Cases with failed download attempts:", len(cases_with_failed_downloads))
if len(cases_with_failed_downloads):
    display(cases_with_failed_downloads.head(100))

print("Cases with nothing to download:", len(cases_with_nothing_to_download))
display(cases_with_nothing_to_download.head(100))

print("Download manifest saved to:", DOWNLOAD_MANIFEST_CSV)
print("Download parquet saved to:", DOWNLOAD_MANIFEST_PARQUET)
print("Download JSONL saved to:", DOWNLOAD_ATTEMPTS_JSONL)
print("Download data directory:", COM_ANTITRUST_DATA_DIR)


Updating JSONL: /home/edik/projects/eccjeu/output/com_antitrust/download_attempts.jsonl
Filesystem check added 3 JSONL already_exists events.
Rows to attempt: 150
Download mode: sequential, one candidate URL at a time, using one browser-like requests.Session.


Download status counts:


download_status
already_exists                     1174
skipped_candidate_after_success    1153
failed_http_status                  148
no_download_available                15
failed_http_202_accepted              2
Name: count, dtype: int64


Download status by source role:


download_status,already_exists,failed_http_202_accepted,failed_http_status,no_download_available,skipped_candidate_after_success
source_role,,,,,
main,593,0,0,0,0
none,0,0,0,15,0
secondary_oj,538,2,148,0,1067
tertiary_oj,43,0,0,0,86


...

Cases with failed download attempts: 17


,case_number,case_title
431,AT.37990,Intel
1772,AT.39914,Euro Interest Rate Derivatives
2260,AT.40401,Second-hand Rolling Stock
2280,AT.40437,Apple - App Store Practices (music streaming)
2306,AT.40512,Euro-denominated bonds (EDB)
2337,AT.40577,Vifor (IV iron products)
2347,AT.40588,Teva Copaxone
2362,AT.40636,SNBB
2382,AT.40642,Pierre Cardin
2392,AT.40669,End-of-life vehicle recycling


Cases with nothing to download: 6


,case_number,case_title,manifest_rows,downloadable_rows
57,AT.36246,Marathon/Ruhrgas/GDF et alia,1,0
66,AT.36533,Yves Saint Laurent,1,0
69,AT.36559,British Gas+Wingas+1,1,0
80,AT.36817,PO/Nederlandse Vissersbond,1,0
284,AT.40054,Ethanol benchmarks,2,0
319,AT.40545,Automotive Starter Batteries,1,0


Download manifest saved to: /home/edik/projects/eccjeu/output/com_antitrust/download_manifest.csv
Download parquet saved to: /home/edik/projects/eccjeu/output/com_antitrust/download_manifest.parquet
Download JSONL saved to: /home/edik/projects/eccjeu/output/com_antitrust/download_attempts.jsonl
Download data directory: /home/edik/projects/eccjeu/data/raw/com_antitrust


## 7b. Enrich consolidated decisions with supporting-document evidence

After downloads, the consolidated decision manifest is rewritten with explicit support columns. These columns show which registry document rows support each decision, which source columns/roles they came from, which source URLs were used, which candidate download URLs were tried, and which local download paths were produced.

In [12]:

# -----------------------------------------------------------------------------
# Enrich consolidated decision manifest with supporting-document evidence
# -----------------------------------------------------------------------------
# This rewrites download_com_antitrust_decisions.csv after the download manifest
# exists, so the decision layer contains both registry evidence and local
# download evidence.


def _join_unique_ordered(values, sep="; "):
    seen = set()
    out = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text.lower() == "nan":
            continue
        for part in re.split(r"\s*;\s*|\n+", text):
            part = part.strip()
            if part and part not in seen:
                seen.add(part)
                out.append(part)
    return sep.join(out)


def _bool_download_success_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(s):
        return s.fillna(False)
    return s.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


def enrich_decision_manifest_with_download_evidence(decision_df: pd.DataFrame, download_df: pd.DataFrame) -> pd.DataFrame:
    decision_df = decision_df.copy()
    download_df = download_df.copy()

    if decision_df.empty or download_df.empty:
        return decision_df

    required = {"decision_key", "item_label", "source_column", "source_role", "source_url", "download_url", "download_path", "download_status"}
    missing = sorted(required - set(download_df.columns))
    if missing:
        raise ValueError(f"download_manifest missing required evidence columns: {missing}")

    if "download_success" not in download_df.columns:
        download_df["download_success"] = False

    downloadable = download_df[download_df["source_role"].fillna("").ne("none")].copy()
    downloadable["download_success_bool"] = _bool_download_success_series(downloadable["download_success"])
    downloadable["downloaded_path_for_support"] = downloadable["download_path"].where(
        downloadable["download_success_bool"]
        | downloadable["download_status"].fillna("").isin({"downloaded", "already_exists"})
    )

    # One JSON object per original source URL. Candidate URLs are nested inside
    # each source item, so EUR-Lex fallback attempts remain auditable without
    # turning the decision CSV into one row per candidate.
    support_rows = []
    group_cols = ["decision_key", "source_url"]
    for (decision_key, source_url), g in downloadable.groupby(group_cols, dropna=False):
        support_rows.append({
            "decision_key": decision_key,
            "source_url": source_url,
            "item_labels": _join_unique_ordered(g["item_label"]),
            "source_columns": _join_unique_ordered(g["source_column"]),
            "source_roles": _join_unique_ordered(g["source_role"]),
            "download_urls": _join_unique_ordered(g["download_url"]),
            "download_url_kinds": _join_unique_ordered(g["download_url_kind"]) if "download_url_kind" in g.columns else "",
            "source_url_candidates_json": _join_unique_ordered(g["source_url_candidates_json"]) if "source_url_candidates_json" in g.columns else "",
            "download_paths": _join_unique_ordered(g["download_path"]),
            "successful_download_paths": _join_unique_ordered(g["downloaded_path_for_support"]),
            "download_statuses": _join_unique_ordered(g["download_status"]),
            "any_downloaded": bool(
                g["download_success_bool"].any()
                or g["download_status"].fillna("").isin({"downloaded", "already_exists"}).any()
            ),
        })

    support_df = pd.DataFrame(support_rows)
    if support_df.empty:
        return decision_df

    # Compact JSON evidence column for manual inspection.
    support_json = (
        support_df.groupby("decision_key", dropna=False)
        .apply(lambda g: json.dumps(
            g[[
                "item_labels", "source_columns", "source_roles", "source_url",
                "download_urls", "download_url_kinds", "source_url_candidates_json", "download_paths", "successful_download_paths",
                "download_statuses", "any_downloaded"
            ]].to_dict("records"),
            ensure_ascii=False
        ))
        .reset_index(name="supporting_documents_json")
    )

    support_agg = (
        support_df.groupby("decision_key", dropna=False)
        .agg(
            supporting_item_labels=("item_labels", _join_unique_ordered),
            supporting_source_columns=("source_columns", _join_unique_ordered),
            supporting_source_roles=("source_roles", _join_unique_ordered),
            supporting_source_urls=("source_url", _join_unique_ordered),
            supporting_download_urls=("download_urls", _join_unique_ordered),
            supporting_download_paths=("download_paths", _join_unique_ordered),
            successful_supporting_download_paths=("successful_download_paths", _join_unique_ordered),
            supporting_download_statuses=("download_statuses", _join_unique_ordered),
            supporting_source_url_count=("source_url", lambda s: len([x for x in s if clean_text(x)])),
            supporting_source_urls_downloaded=("any_downloaded", "sum"),
        )
        .reset_index()
        .merge(support_json, on="decision_key", how="left")
    )

    # Avoid duplicate columns if this cell is rerun.
    evidence_cols = [c for c in support_agg.columns if c != "decision_key"]
    decision_df = decision_df.drop(columns=[c for c in evidence_cols if c in decision_df.columns], errors="ignore")
    decision_df = decision_df.merge(support_agg, on="decision_key", how="left")

    for col in evidence_cols:
        if col not in {"supporting_source_url_count", "supporting_source_urls_downloaded"}:
            decision_df[col] = decision_df[col].fillna("")
    for col in ["supporting_source_url_count", "supporting_source_urls_downloaded"]:
        if col in decision_df.columns:
            decision_df[col] = pd.to_numeric(decision_df[col], errors="coerce").fillna(0).astype(int)

    decision_df["supporting_source_urls_missing"] = (
        decision_df.get("supporting_source_url_count", 0)
        - decision_df.get("supporting_source_urls_downloaded", 0)
    ).astype(int)

    return decision_df


decision_manifest = enrich_decision_manifest_with_download_evidence(decision_manifest, download_manifest)
decision_manifest.to_csv(DOWNLOAD_COM_ANTITRUST_DECISIONS_CSV, index=False, encoding="utf-8-sig")

print("Re-saved consolidated decision manifest with supporting-document evidence:")
print(" -", DOWNLOAD_COM_ANTITRUST_DECISIONS_CSV)

evidence_cols_preview = [
    "Decision identifier", "decision_key", "case_number", "decision_date", "decision_label",
    "supporting_item_labels", "supporting_source_columns", "supporting_source_roles",
    "supporting_source_urls_downloaded", "supporting_source_url_count",
    "supporting_source_urls_missing", "successful_supporting_download_paths",
]
display(decision_manifest[[c for c in evidence_cols_preview if c in decision_manifest.columns]].head(20))


/tmp/ipykernel_506059/2343456004.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return s.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
/tmp/ipykernel_506059/2343456004.py:85: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: json.dumps(


Re-saved consolidated decision manifest with supporting-document evidence:
 - /home/edik/projects/eccjeu/output/com_antitrust/download_com_antitrust_decisions.csv


,Decision identifier,decision_key,case_number,decision_date,decision_label,supporting_item_labels,supporting_source_columns,supporting_source_roles,supporting_source_urls_downloaded,supporting_source_url_count,supporting_source_urls_missing,successful_supporting_download_paths
0,COMDEC00001,AT.28841+19-04-1977,AT.28841,19/04/1977,Prohibition Decision,Prohibition Decision,Item link,main,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
1,COMDEC00002,AT.29629+14-12-1982,AT.29629,14/12/1982,Prohibition Decision,Prohibition Decision,Item link,main,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
2,COMDEC00003,AT.30373+12-04-1999,AT.30373,12/04/1999,Old milestones - Exemption with condition deci...,Old milestones - Exemption with condition deci...,Item OJ link,secondary_oj,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
3,COMDEC00004,AT.32150+10-05-2000,AT.32150,10/05/2000,Old milestones - Exemption with condition deci...,Old milestones - Exemption with condition deci...,Item OJ link,secondary_oj,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
4,COMDEC00005,AT.32450+30-04-2004,AT.32450,30/04/2004,Prohibition Decision,Prohibition Decision,Item link,main,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
5,COMDEC00006,AT.32948+21-12-1994,AT.32948,21/12/1994,Prohibition Decision,Prohibition Decision,Item OJ link,secondary_oj,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
6,COMDEC00007,AT.33585+25-11-1992,AT.33585,25/11/1992,Prohibition Decision,Prohibition Decision,Item link,main,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
7,COMDEC00008,AT.33708+14-10-1998,AT.33708,14/10/1998,Prohibition Decision,Prohibition Decision,Item link,main,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
8,COMDEC00009,AT.33884+26-10-1999,AT.33884,26/10/1999,Prohibition Decision,Prohibition Decision,Item OJ link,secondary_oj,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...
9,COMDEC00010,AT.34018+16-05-2000,AT.34018,16/05/2000,Prohibition Decision,Prohibition Decision,Item OJ link,secondary_oj,1,1,0,/home/edik/projects/eccjeu/data/raw/com_antitr...


## 8. Decision-level and case-level validation outputs

This section projects the URL-level download manifest back up to two review layers:

1. **Decision/document level**: one row per relevant row in `download_com_antitrust_documents.csv`, distinguishing the main Commission URL (`Item link`) from secondary OJ/EUR-Lex URLs (`Item OJ link`).
2. **Case level**: one row per case, aggregating all relevant decision/document rows and showing whether the case is relevant and how complete its downloads are.

It writes:

- `validate_com_antitrust_decisions.csv`
- `validate_com_antitrust_cases.csv`

Completeness is calculated at the original **source URL** level, not at the expanded candidate URL level. So if one EUR-Lex candidate succeeds for a source URL, that source URL is counted as downloaded.


In [13]:

# -----------------------------------------------------------------------------
# Decision-level and case-level validation manifests
# -----------------------------------------------------------------------------
# Inputs expected from earlier cells:
# - view1: case-level manifest
# - view2: document/decision-level manifest
# - decision_manifest: consolidated decision manifest, one row per case + date
# - download_manifest: URL/candidate-level download manifest
#
# Outputs:
# - validate_com_antitrust_decisions.csv
# - validate_com_antitrust_cases.csv
#
# v6 change:
# Validation is now based on the consolidated decision manifest, not directly on
# raw View 2 rows. This means decision completeness follows the rule:
# "one case cannot have multiple decisions on the same date"; same case + same
# date is one decision unless conflicting labels make it an Unclear Decision.

VALIDATE_DECISIONS_CSV = COM_ANTITRUST_OUTPUT_DIR / "validate_com_antitrust_decisions.csv"
VALIDATE_CASES_CSV = COM_ANTITRUST_OUTPUT_DIR / "validate_com_antitrust_cases.csv"


def _as_bool(x):
    """Robust boolean conversion for bools, strings, ints, and missing values."""
    if isinstance(x, pd.Series):
        if pd.api.types.is_bool_dtype(x):
            return x.fillna(False)
        return x.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
    return str(x).strip().lower() in {"true", "1", "yes", "y"}


def _successful_candidate_status(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip().isin({"downloaded", "already_exists"})


def _join_unique(values) -> str:
    vals = []
    for value in values:
        if pd.isna(value):
            continue
        text = str(value).strip()
        if text and text.lower() != "nan":
            vals.append(text)
    return "; ".join(sorted(set(vals)))




def normalize_decision_key_part(value):
    text = str(value).strip() if not pd.isna(value) else ""
    text = re.sub(r"\s+", "", text)
    text = text.replace("/", "-")
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    return text or "missing"


def make_decision_key(case_number, decision_date):
    return f"{normalize_decision_key_part(case_number)}+{normalize_decision_key_part(decision_date)}"


def make_decision_key_reverse(case_number, decision_date):
    return f"{normalize_decision_key_part(decision_date)}_{normalize_decision_key_part(case_number)}"


def _source_role_from_column(source_column: str) -> str:
    return {
        "Item link": "main",
        "Item OJ link": "secondary_oj",
        "Item tertiary OJ link": "tertiary_oj",
    }.get(str(source_column).strip(), "other")


def _source_level_download_manifest(download_manifest: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse candidate rows to one row per original source URL.

    The download manifest can have multiple rows per source URL because one source
    URL may produce several candidates, especially EUR-Lex HTML/PDF fallbacks.
    Validation should count the source URL as downloaded if any candidate worked.
    """
    dm = download_manifest.copy()

    required = {
        "identifier", "case_number", "case_title", "initiation_of_proceedings",
        "item_type", "item_date", "item_label", "relevant", "source_column",
        "source_url", "download_url", "download_status",
    }
    missing = sorted(required - set(dm.columns))
    if missing:
        raise ValueError(f"download_manifest missing required columns for validation: {missing}")

    dm["relevant_bool"] = _as_bool(dm["relevant"])
    dm["candidate_success"] = _as_bool(dm.get("download_success", False)) | _successful_candidate_status(dm["download_status"])
    dm["candidate_local_exists"] = _as_bool(dm.get("local_file_exists", False))

    if "source_role" not in dm.columns:
        dm["source_role"] = dm["source_column"].map(_source_role_from_column)
    else:
        dm["source_role"] = dm["source_role"].fillna("")
        empty_role = dm["source_role"].astype(str).str.strip().eq("")
        dm.loc[empty_role, "source_role"] = dm.loc[empty_role, "source_column"].map(_source_role_from_column)

    # Empty source_url rows are explicit no-download rows. Keep them out of
    # source-url completeness counts; the absence of URLs is handled from the
    # consolidated decision manifest.
    dm_url = dm[dm["source_url"].fillna("").astype(str).str.strip().ne("")].copy()

    if dm_url.empty:
        return pd.DataFrame(columns=[
            "decision_key", "decision_key_reverse", "case_number", "item_date", "source_role", "source_url",
            "source_downloaded", "source_local_exists", "candidate_rows",
            "successful_candidate_rows", "source_statuses", "downloaded_paths",
            "http_statuses", "errors"
        ])

    dm_url["decision_key"] = dm_url.apply(
        lambda r: make_decision_key(r.get("case_number", ""), r.get("item_date", "")),
        axis=1,
    )
    dm_url["decision_key_reverse"] = dm_url.apply(
        lambda r: make_decision_key_reverse(r.get("case_number", ""), r.get("item_date", "")),
        axis=1,
    )

    source_key_cols = [
        "decision_key", "decision_key_reverse", "case_number", "item_date", "source_role", "source_column", "source_url",
    ]

    source_level = (
        dm_url.groupby(source_key_cols, dropna=False)
              .agg(
                  source_downloaded=("candidate_success", "any"),
                  source_local_exists=("candidate_local_exists", "any"),
                  candidate_rows=("download_url", "size"),
                  successful_candidate_rows=("candidate_success", "sum"),
                  source_statuses=("download_status", _join_unique),
                  downloaded_paths=("download_path", _join_unique),
                  http_statuses=("http_status", _join_unique),
                  errors=("error", _join_unique),
              )
              .reset_index()
    )
    return source_level


def _decision_download_status(row: pd.Series) -> str:
    main_total = int(row.get("main_source_urls", 0) or 0)
    main_done = int(row.get("main_source_urls_downloaded", 0) or 0)
    secondary_total = int(row.get("secondary_oj_source_urls", 0) or 0)
    secondary_done = int(row.get("secondary_oj_source_urls_downloaded", 0) or 0)
    tertiary_total = int(row.get("tertiary_oj_source_urls", 0) or 0)
    tertiary_done = int(row.get("tertiary_oj_source_urls_downloaded", 0) or 0)

    if main_total > 0:
        if main_done == 0:
            return "main_not_downloaded"
        if secondary_total > 0:
            if secondary_done >= secondary_total:
                return "main_and_secondary_downloaded"
            return "main_downloaded_secondary_incomplete"
        return "main_downloaded_no_secondary"

    if secondary_total > 0:
        if secondary_done >= secondary_total:
            return "secondary_only_downloaded"
        if secondary_done > 0:
            return "secondary_only_partial"
        return "secondary_only_not_downloaded"

    if tertiary_total > 0:
        if tertiary_done >= tertiary_total:
            return "tertiary_oj_found_downloaded"
        if tertiary_done > 0:
            return "tertiary_oj_found_partial"
        return "tertiary_oj_found_not_downloaded"

    return "no_url_available"


def build_validate_com_antitrust_decisions(
    decision_manifest: pd.DataFrame,
    download_manifest: pd.DataFrame,
) -> pd.DataFrame:
    decisions = decision_manifest.copy()

    if "decision_key" not in decisions.columns:
        decisions["decision_key"] = decisions.apply(
            lambda r: make_decision_key(r.get("case_number", ""), r.get("decision_date", "")),
            axis=1,
        )
    if "decision_key_reverse" not in decisions.columns:
        decisions["decision_key_reverse"] = decisions.apply(
            lambda r: make_decision_key_reverse(r.get("case_number", ""), r.get("decision_date", "")),
            axis=1,
        )

    source_level = _source_level_download_manifest(download_manifest)

    if source_level.empty:
        grouped = pd.DataFrame(columns=["decision_key"])
    else:
        grouped = (
            source_level.groupby("decision_key", dropna=False)
                        .agg(
                            source_urls=("source_url", "nunique"),
                            downloaded_source_urls=("source_downloaded", "sum"),
                            main_source_urls=("source_role", lambda s: int((s == "main").sum())),
                            main_source_urls_downloaded=("source_downloaded", lambda s: 0),  # filled below
                            secondary_oj_source_urls=("source_role", lambda s: int((s == "secondary_oj").sum())),
                            secondary_oj_source_urls_downloaded=("source_downloaded", lambda s: 0),  # filled below
                            tertiary_oj_source_urls=("source_role", lambda s: int((s == "tertiary_oj").sum())),
                            tertiary_oj_source_urls_downloaded=("source_downloaded", lambda s: 0),  # filled below
                            downloaded_paths=("downloaded_paths", _join_unique),
                            source_statuses=("source_statuses", _join_unique),
                            http_statuses=("http_statuses", _join_unique),
                            errors=("errors", _join_unique),
                        )
                        .reset_index()
        )

        # The role-specific downloaded counts are easiest and least error-prone
        # to compute separately.
        role_done = (
            source_level[source_level["source_downloaded"]]
            .groupby(["decision_key", "source_role"], dropna=False)
            .size()
            .unstack(fill_value=0)
            .reset_index()
        )
        for role_col, out_col in [
            ("main", "main_source_urls_downloaded"),
            ("secondary_oj", "secondary_oj_source_urls_downloaded"),
            ("tertiary_oj", "tertiary_oj_source_urls_downloaded"),
        ]:
            if role_col not in role_done.columns:
                role_done[role_col] = 0
            role_done = role_done.rename(columns={role_col: out_col})

        keep_cols = ["decision_key", "main_source_urls_downloaded", "secondary_oj_source_urls_downloaded", "tertiary_oj_source_urls_downloaded"]
        role_done = role_done[[c for c in keep_cols if c in role_done.columns]]
        grouped = grouped.drop(columns=[c for c in keep_cols if c in grouped.columns and c != "decision_key"], errors="ignore")
        grouped = grouped.merge(role_done, on="decision_key", how="left")

    out = decisions.merge(grouped, on="decision_key", how="left")

    count_cols = [
        "source_urls", "downloaded_source_urls",
        "main_source_urls", "main_source_urls_downloaded",
        "secondary_oj_source_urls", "secondary_oj_source_urls_downloaded",
        "tertiary_oj_source_urls", "tertiary_oj_source_urls_downloaded",
    ]
    for col in count_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0).astype(int)

    for col in ["downloaded_paths", "source_statuses", "http_statuses", "errors"]:
        if col not in out.columns:
            out[col] = ""
        out[col] = out[col].fillna("")

    out["download_status"] = out.apply(_decision_download_status, axis=1)
    out["available_urls_downloaded"] = out["source_urls"].eq(out["downloaded_source_urls"])
    out["has_any_download"] = out["downloaded_source_urls"].gt(0)
    out["has_main_download"] = out["main_source_urls_downloaded"].gt(0)
    out["has_secondary_oj_download"] = out["secondary_oj_source_urls_downloaded"].gt(0)
    out["has_tertiary_oj_download"] = out["tertiary_oj_source_urls_downloaded"].gt(0)

    # Keep compact status columns near the front.
    front = [
        "Decision identifier", "decision_key", "decision_key_reverse", "case_number", "case_title",
        "decision_date", "decision_label", "is_unclear_decision",
        "download_status", "url_availability", "source_urls", "downloaded_source_urls",
        "main_source_urls", "main_source_urls_downloaded",
        "secondary_oj_source_urls", "secondary_oj_source_urls_downloaded",
        "tertiary_oj_source_urls", "tertiary_oj_source_urls_downloaded",
        "available_urls_downloaded", "has_any_download", "has_main_download",
        "has_secondary_oj_download", "has_tertiary_oj_download",
    ]
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]


def _case_download_status(row: pd.Series) -> str:
    n = int(row.get("relevant_decisions", 0) or 0)
    if n == 0:
        return "not_relevant"

    no_url = int(row.get("no_url_available_decisions", 0) or 0)
    complete = int(row.get("complete_download_decisions", 0) or 0)
    partial = int(row.get("partial_or_incomplete_decisions", 0) or 0)
    tertiary = int(row.get("tertiary_oj_decisions", 0) or 0)

    if no_url == n:
        return "no_url_available"
    if complete == n:
        if tertiary > 0:
            return "all_relevant_decisions_downloaded_with_tertiary_oj"
        return "all_relevant_decisions_downloaded"
    if complete + no_url == n:
        return "all_available_urls_downloaded_some_decisions_no_url"
    if complete > 0 or partial > 0:
        return "partially_downloaded"
    return "not_downloaded"


def build_validate_com_antitrust_cases(
    view1_df: pd.DataFrame,
    validate_decisions: pd.DataFrame,
) -> pd.DataFrame:
    cases = view1_df.copy()

    if validate_decisions.empty:
        cases["relevant_case"] = False
        cases["relevant_decisions"] = 0
        cases["case_download_status"] = "not_relevant"
        return cases

    vd = validate_decisions.copy()
    vd["complete_decision_download"] = vd["download_status"].isin({
        "main_downloaded_no_secondary",
        "main_and_secondary_downloaded",
        "secondary_only_downloaded",
        "tertiary_oj_found_downloaded",
    })
    vd["partial_or_incomplete_decision"] = vd["download_status"].isin({
        "main_downloaded_secondary_incomplete",
        "secondary_only_partial",
        "tertiary_oj_found_partial",
    })
    vd["no_url_decision"] = vd["download_status"].eq("no_url_available")
    vd["tertiary_oj_decision"] = vd["download_status"].astype(str).str.startswith("tertiary_oj_found")
    vd["unclear_decision"] = _as_bool(vd.get("is_unclear_decision", False))

    grouped = (
        vd.groupby("case_number", dropna=False)
          .agg(
              relevant_decisions=("decision_key", "nunique"),
              decision_dates=("decision_date", _join_unique),
              decision_labels=("decision_labels", _join_unique),
              decision_download_statuses=("download_status", _join_unique),
              complete_download_decisions=("complete_decision_download", "sum"),
              partial_or_incomplete_decisions=("partial_or_incomplete_decision", "sum"),
              no_url_available_decisions=("no_url_decision", "sum"),
              tertiary_oj_decisions=("tertiary_oj_decision", "sum"),
              unclear_decisions=("unclear_decision", "sum"),
              source_urls=("source_urls", "sum"),
              downloaded_source_urls=("downloaded_source_urls", "sum"),
              downloaded_paths=("downloaded_paths", _join_unique),
              errors=("errors", _join_unique),
          )
          .reset_index()
    )

    out = cases.merge(grouped, left_on="Case number", right_on="case_number", how="left")
    out = out.drop(columns=["case_number"], errors="ignore")

    numeric_cols = [
        "relevant_decisions", "complete_download_decisions",
        "partial_or_incomplete_decisions", "no_url_available_decisions",
        "tertiary_oj_decisions", "unclear_decisions", "source_urls",
        "downloaded_source_urls",
    ]
    for col in numeric_cols:
        if col not in out.columns:
            out[col] = 0
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0).astype(int)

    for col in ["decision_dates", "decision_labels", "decision_download_statuses", "downloaded_paths", "errors"]:
        if col not in out.columns:
            out[col] = ""
        out[col] = out[col].fillna("")

    out["relevant_case"] = out["relevant_decisions"].gt(0)
    out["case_download_status"] = out.apply(_case_download_status, axis=1)

    front = [
        "Case number", "Case title", "relevant_case", "case_download_status",
        "relevant_decisions", "complete_download_decisions",
        "partial_or_incomplete_decisions", "no_url_available_decisions",
        "tertiary_oj_decisions", "unclear_decisions",
        "source_urls", "downloaded_source_urls", "decision_dates",
        "decision_labels", "decision_download_statuses",
    ]
    front = [c for c in front if c in out.columns]
    rest = [c for c in out.columns if c not in front]
    return out[front + rest]


# Rebuild the consolidated decision manifest if the earlier cell was not run.
if "decision_manifest" not in globals():
    decision_manifest = build_com_antitrust_decisions(view2)
    decision_manifest.to_csv(DOWNLOAD_COM_ANTITRUST_DECISIONS_CSV, index=False, encoding="utf-8-sig")

validate_decisions = build_validate_com_antitrust_decisions(decision_manifest, download_manifest)
validate_cases = build_validate_com_antitrust_cases(view1, validate_decisions)

validate_decisions.to_csv(VALIDATE_DECISIONS_CSV, index=False, encoding="utf-8-sig")
validate_cases.to_csv(VALIDATE_CASES_CSV, index=False, encoding="utf-8-sig")

print("Saved validation CSVs:")
print(" -", VALIDATE_DECISIONS_CSV)
print(" -", VALIDATE_CASES_CSV)

print("\nDecision-level download status:")
display(validate_decisions["download_status"].value_counts(dropna=False))

print("\nDecision URL availability:")
display(validate_decisions["url_availability"].value_counts(dropna=False))

print("\nCase-level download status:")
display(validate_cases["case_download_status"].value_counts(dropna=False))

print("\nCore validation summary:")
summary = {
    "relevant_decisions": len(validate_decisions),
    "unclear_decisions": int(validate_decisions.get("is_unclear_decision", pd.Series(dtype=bool)).sum()),
    "decisions_with_main_url": int((validate_decisions["main_source_urls"] > 0).sum()),
    "decisions_with_secondary_only_url": int(((validate_decisions["main_source_urls"] == 0) & (validate_decisions["secondary_oj_source_urls"] > 0)).sum()),
    "decisions_with_tertiary_oj_only_url": int(((validate_decisions["main_source_urls"] == 0) & (validate_decisions["secondary_oj_source_urls"] == 0) & (validate_decisions["tertiary_oj_source_urls"] > 0)).sum()),
    "decisions_with_no_url": int(validate_decisions["download_status"].eq("no_url_available").sum()),
    "relevant_cases": int(validate_cases["relevant_case"].sum()),
}
display(pd.Series(summary).to_frame("value"))


Saved validation CSVs:
 - /home/edik/projects/eccjeu/output/com_antitrust/validate_com_antitrust_decisions.csv
 - /home/edik/projects/eccjeu/output/com_antitrust/validate_com_antitrust_cases.csv

Decision-level download status:


/tmp/ipykernel_506059/2664313256.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return x.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})
/tmp/ipykernel_506059/2664313256.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return x.fillna(False).astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y"})


download_status
main_and_secondary_downloaded           149
main_downloaded_no_secondary            123
secondary_only_downloaded                80
main_downloaded_secondary_incomplete     18
tertiary_oj_found_downloaded             16
no_url_available                         14
Name: count, dtype: int64


Decision URL availability:


url_availability
main_url_available        290
secondary_oj_available     80
tertiary_oj_found          16
no_url_available           14
Name: count, dtype: int64


Case-level download status:


case_download_status
not_relevant                                           414
all_relevant_decisions_downloaded                      291
partially_downloaded                                    17
all_relevant_decisions_downloaded_with_tertiary_oj      16
all_available_urls_downloaded_some_decisions_no_url      7
no_url_available                                         6
Name: count, dtype: int64


Core validation summary:


,value
relevant_decisions,400
unclear_decisions,4
decisions_with_main_url,290
decisions_with_secondary_only_url,80
decisions_with_tertiary_oj_only_url,16
decisions_with_no_url,14
relevant_cases,337


## General notes on the code

This notebook is intentionally written as a transparent data-transformation script rather than as a highly abstract package. The most important design choice is to keep the two analytical levels separate:

- the **case-level view** is useful for reading and checking one case at a time;
- the **document-level view** is useful for systematic filtering, classification, and later manual review.

The current implementation focuses on **exploding and organising** the Commission JSON into interpretable spreadsheet views. It does not yet classify relevance or contestability. That is intentional: contestability depends on legal context and sometimes on the exact document contents. The notebook therefore prepares a clean manifest first, so that manual review or rule-based classification can be added afterwards.

The rule-based relevance column is intentionally narrow. Rejection-of-complaint decisions, trustee material, market tests, advisory/hearing-officer material, initiation documents, summaries, press releases, proposed commitments, and closure/administrative closure rows are not flagged for download.

The EUR-Lex URL helper deliberately avoids treating Official Journal references such as `C:2008:021:5` as CELEX identifiers. Instead, it builds `uriserv:OJ...` links or falls back to the OJ table of contents when references are incomplete.
